
# Function Approximator — **Symbolic Regression (GPU)**
## v6 — AOS + Derivative Loss + Hints + Macros + Curriculum + Subsampling + Memetic
**Backends:** PyTorch (CUDA si disponible) o CuPy (RTX).  
Este cuaderno incorpora aceleradores para encontrar la forma de la función **más rápido** cuando:
- Tienes **derivadas** \(y'(x)\) y/o \(y''(x)\).
- Tienes una **función parecida** ("hint") para sembrar la población.
- Quieres **auto‑selección de operadores** (AOS), **macros**, **currículum**, **submuestreo** y **búsqueda local** (memetic).


In [ ]:

import random, time, importlib
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

def has(module_name): 
    try:
        return importlib.util.find_spec(module_name) is not None
    except Exception:
        return False

# ===== BACKEND: "torch" o "cupy" =====
BACKEND = "torch"   # Cambia a "cupy" si prefieres CuPy
print("BACKEND =", BACKEND)

# ===== Flags de rendimiento (Fase 2) =====
BATCH_EVAL = True      # evalúa la población por lotes en GPU (una sincronización por lote)
FITNESS_CACHE = True   # memoiza fitness por individuo y generación en la ruta clásica (BATCH_EVAL=False)
USE_TF32 = True        # TF32 en GPUs Ampere+ (solo afecta a matmul; inocuo en estas ops)
DTYPE = "float32"      # "float32" o "float64" para la evaluación en GPU
SIMPLIFY_EVERY = 0     # simplificación algebraica ligera de la población cada N generaciones (0 = off)
DEDUP_EVERY = 0        # deduplicación semántica cada N generaciones (0 = off)
AOS_UPDATE_CHUNK = 1   # hijos por sub-lote entre actualizaciones AOS.
                       #  1  = recompensas inmediatas por hijo, como el original (mismo algoritmo;
                       #       la trayectoria puede divergir tras decenas de generaciones por
                       #       diferencias de +-1 ulp en el orden de reducción del MSE en lotes)
                       #  16 = máxima velocidad (~x16): recompensas levemente diferidas,
                       #       misma calidad media pero dinámica AOS menos reactiva

# ===== Flags de calidad de búsqueda (Fase 3) =====
AOS_XOVER_REWARD = True   # recompensa AOS también al crossover (ops del subárbol donado)
AOS_REWARD_NORM  = True   # normaliza las recompensas por la varianza de Y (escala-invariante)
SELECTION = "alpha"       # "alpha": mse + alpha*n_nodos | "nsga2": frente de Pareto precisión-complejidad
PRETTY_CONSTANTS = True   # to_sympy reconoce enteros/fracciones/pi/E con tolerancia estricta

# ===== Operador EML (Fase 4) — eml(x,y) = exp(x) - ln(y) =====
# A. Odrzywolek, "All elementary functions from a single binary operator"
# (arXiv:2603.21852): eml genera todas las funciones elementales y el paper lo
# propone para regresión simbólica (sección 4.3).
EML_MODE = "off"          # "off" | "operator" (eml en el catálogo, ponderable por el AOS) | "pure"
EML_SYMPY_EXPAND = True   # to_sympy: True = forma exp/ln expandida (evaluable); False = eml(u,v) compacta
EML_PURE_DEPTH  = 4       # profundidad de la estructura completa del modo puro (sin init evolutiva)
EML_PURE_STARTS = 256     # multi-starts entrenados EN PARALELO (dimensión extra del tensor, no bucle)
EML_PURE_STEPS  = 1500    # pasos de Adam del modo puro
EML_PURE_LR     = 5e-3

if BACKEND == "torch":
    import torch
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    print("PyTorch device:", device)
    if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
    if USE_TF32:
        torch.set_float32_matmul_precision("high")  # Fase 2.3
elif BACKEND == "cupy":
    if not has("cupy"):
        raise RuntimeError("No se encontró CuPy. Instala: pip install cupy-cuda12x  (o cupy-cuda11x)")
    import cupy as cp
    import cupyx.scipy.special as cpsp
    print("CuPy device: CUDA")
    SCIPY_AVAILABLE = has("scipy")
    if SCIPY_AVAILABLE:
        from scipy.optimize import minimize
        print("SciPy encontrado: habilitado fine-tuning (Nelder–Mead)")
    else:
        print("SciPy no encontrado: fine-tuning se omitirá en backend CuPy")
else:
    raise ValueError("BACKEND debe ser 'torch' o 'cupy'")

SEED=2   # semilla de la demo (recupera el caso de prueba con la config por defecto;
         # la tasa de recuperación multi-semilla se reporta en los benchmarks de la Fase 5)
random.seed(SEED); np.random.seed(SEED)
# Reproducibilidad también en el backend GPU (Fase 1.3)
if BACKEND == "torch":
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
else:
    cp.random.seed(SEED)

# ===== Guardas numéricas y helpers de backend =====
def finfo_bounds(x):
    if BACKEND == "torch":
        f = torch.finfo(x.dtype)
        return f.min*0.25, f.max*0.25
    else:
        f = cp.finfo(x.dtype)
        return f.min*0.25, f.max*0.25

def clip_safe(x, lo=None, hi=None):
    lo_f, hi_f = finfo_bounds(x)
    if lo is None: lo = lo_f
    if hi is None: hi = hi_f
    lo = max(lo_f, min(float(lo), hi_f)); hi = min(hi_f, max(float(hi), lo_f))
    if BACKEND == "torch": return torch.clamp(x, min=float(lo), max=float(hi))
    else:                  return cp.clip(x, float(lo), float(hi))

def nan_to_num_safe(x, pos=None, neg=None):
    lo_f, hi_f = finfo_bounds(x)
    if pos is None: pos = hi_f
    if neg is None: neg = lo_f
    if BACKEND == "torch": return torch.nan_to_num(x, nan=0.0, posinf=float(pos), neginf=float(neg))
    else:                  return cp.nan_to_num(x, nan=0.0, posinf=float(pos), neginf=float(neg))

def exp_safe(x):
    if BACKEND == "torch":
        max_in = 80.0 if x.dtype == torch.float32 else 700.0
        y = torch.exp(clip_safe(x, -max_in, max_in))
        return clip_safe(nan_to_num_safe(y))
    else:
        max_in = 80.0 if x.dtype == cp.float32 else 700.0
        y = cp.exp(clip_safe(x, -max_in, max_in))
        return clip_safe(nan_to_num_safe(y))

def to_device(arr):
    if BACKEND == "torch":
        dt = torch.float64 if DTYPE == "float64" else torch.float32
        return torch.as_tensor(arr, dtype=dt, device=device)
    else:
        dt = cp.float64 if DTYPE == "float64" else cp.float32
        return cp.asarray(arr, dtype=dt)

def to_numpy(x):
    if BACKEND == "torch": return x.detach().cpu().numpy()
    else:                  return cp.asnumpy(x)

def ones_like(x): return torch.ones_like(x) if BACKEND=="torch" else cp.ones_like(x)
def abs_(x):      return torch.abs(x)       if BACKEND=="torch" else cp.abs(x)
def sin_(x):      return torch.sin(x)       if BACKEND=="torch" else cp.sin(x)
def cos_(x):      return torch.cos(x)       if BACKEND=="torch" else cp.cos(x)

def tan_(x):
    x = clip_safe(x, -50.0, 50.0)
    y = torch.tan(x) if BACKEND=="torch" else cp.tan(x)
    return clip_safe(nan_to_num_safe(y))

def log_(x):   return (torch.log if BACKEND=="torch" else cp.log)(abs_(x) + (1e-8))
def sqrt_(x):  return (torch.sqrt if BACKEND=="torch" else cp.sqrt)(abs_(x) + (1e-8))
def tanh_(x):  return (torch.tanh if BACKEND=="torch" else cp.tanh)(x)
def sigmoid_(x):
    if BACKEND=="torch": return torch.sigmoid(x)
    else:                return 1/(1+cp.exp(-x))
def asin_(x):  return (torch.asin if BACKEND=="torch" else cp.arcsin)(clip_safe(x, -1+1e-6, 1-1e-6))
def acos_(x):  return (torch.acos if BACKEND=="torch" else cp.arccos)(clip_safe(x, -1+1e-6, 1-1e-6))
def atan_(x):  return (torch.atan if BACKEND=="torch" else cp.arctan)(x)

def add_(a,b): return a+b
def sub_(a,b): return a-b
def mul_(a,b): return a*b
def div_(a,b, eps=1e-8):
    if BACKEND=="torch":
        b = torch.where(torch.abs(b) < eps, torch.sign(b)*eps + (b==0)*eps, b)
    else:
        b = cp.where(cp.abs(b) < eps, cp.sign(b)*eps + (b==0)*eps, b)
    return clip_safe(nan_to_num_safe(a/b))

def pow_(a,b):
    a = clip_safe(a, -1e3, 1e3); b = clip_safe(b, -5, 5)
    if BACKEND=="torch":
        out = torch.pow(a,b)
    else:
        out = cp.power(a,b)
    return clip_safe(nan_to_num_safe(out))

def eml_(a, b):
    # eml(a,b) = exp(a) - ln(b), con las mismas guardas que exp_safe y log_
    # (exp recortado contra overflow, log de |b|+1e-8); diferenciable en torch
    return exp_safe(a) - log_(b)

# ===== Especiales =====
def gamma_(x):
    if BACKEND=="torch":
        z = clip_safe(x, -170.0, 170.0)
        return clip_safe(nan_to_num_safe(exp_safe(torch.lgamma(z))))
    else:
        z = clip_safe(x, -170.0, 170.0)
        return clip_safe(nan_to_num_safe(exp_safe(cpsp.gammaln(z))))

def beta_(a,b):
    if BACKEND=="torch":
        a = clip_safe(a, 1e-6, 170.0); b = clip_safe(b, 1e-6, 170.0)
        return clip_safe(exp_safe(torch.lgamma(a)+torch.lgamma(b)-torch.lgamma(a+b)), 0.0, None)
    else:
        a = clip_safe(a, 1e-6, 170.0); b = clip_safe(b, 1e-6, 170.0)
        return clip_safe(nan_to_num_safe(exp_safe(cpsp.gammaln(a)+cpsp.gammaln(b)-cpsp.gammaln(a+b))), 0.0, None)

def erf_(x):
    if BACKEND=="torch": 
        return torch.special.erf(x)
    else:
        return cpsp.erf(x)

def erfc_(x):
    if BACKEND=="torch": 
        return torch.special.erfc(x)
    else:
        return cpsp.erfc(x)

def j0_(x):
    if BACKEND=="torch" and hasattr(torch.special,"j0"): return torch.special.j0(x)
    if BACKEND=="cupy": return cpsp.j0(x)
    import numpy as _np
    try:
        import scipy.special as _sc; vals=_sc.j0(to_numpy(x))
    except Exception: vals=_np.zeros_like(to_numpy(x), dtype=_np.float32)
    return to_device(vals)

def j1_(x):
    if BACKEND=="torch" and hasattr(torch.special,"j1"): return torch.special.j1(x)
    if BACKEND=="cupy": return cpsp.j1(x)
    import numpy as _np
    try:
        import scipy.special as _sc; vals=_sc.j1(to_numpy(x))
    except Exception: vals=_np.zeros_like(to_numpy(x), dtype=_np.float32)
    return to_device(vals)

def y0_(x):
    x = clip_safe(x, 1e-6, None)
    if BACKEND=="cupy": return cpsp.y0(x)
    import numpy as _np
    try:
        import scipy.special as _sc; vals=_sc.y0(to_numpy(x))
    except Exception: vals=_np.zeros_like(to_numpy(x), dtype=_np.float32)
    return to_device(vals)

def y1_(x):
    x = clip_safe(x, 1e-6, None)
    if BACKEND=="cupy": return cpsp.y1(x)
    import numpy as _np
    try:
        import scipy.special as _sc; vals=_sc.y1(to_numpy(x))
    except Exception: vals=_np.zeros_like(to_numpy(x), dtype=_np.float32)
    return to_device(vals)

def chebyshevT_(x, n):
    if n==0: return ones_like(x)
    if n==1: return x
    T0 = ones_like(x); T1 = x
    for k in range(2, n+1): T0, T1 = T1, 2.0*x*T1 - T0
    return T1

def legendreP_(x, n):
    if n==0: return ones_like(x)
    if n==1: return x
    P0 = ones_like(x); P1 = x
    for k in range(2, n+1): P0, P1 = P1, ((2*k-1)*x*P1 - (k-1)*P0)/k
    return P1




## Árbol de expresión y operadores


In [ ]:

class Operation:
    def __init__(self, name=None, o1=None, o2=None):
        self.name=name; self.o1=o1; self.o2=o2
    def eval_backend(self, x, const_map=None): raise NotImplementedError
    def to_text(self): raise NotImplementedError
    def to_sympy(self, xs): raise NotImplementedError  # conversión recursiva nodo -> SymPy (Fase 1.1)
    def clone(self): raise NotImplementedError

class Constant(Operation):
    def __init__(self, value=None): 
        super().__init__(name="const")
        if value is None: value = random.uniform(-5,5)
        self.value=float(value); self._id=id(self)
    def eval_backend(self, x, const_map=None):
        if const_map is not None and self._id in const_map:
            c = const_map[self._id]
            return c + (0*c)
        return (x*0)+self.value
    def to_text(self):
        v=self.value
        if not np.isfinite(v): return f"{v}"
        return str(int(round(v))) if abs(v-round(v))<1e-10 else f"{v:.5g}"
    def to_sympy(self, xs):
        v = self.value
        if not np.isfinite(v): return sp.nan
        if abs(v - round(v)) < 1e-10: return sp.Integer(round(v))
        if PRETTY_CONSTANTS:
            # reconocimiento estricto (1e-9) de fracciones simples y múltiplos de pi/E
            try:
                fr = sp.Rational(v).limit_denominator(12)
                if abs(float(fr) - v) < 1e-9: return fr
                cand = sp.nsimplify(v, [sp.pi, sp.E], tolerance=1e-9, rational=False)
                if cand.is_number and abs(float(cand) - v) < 1e-9 and len(str(cand)) <= 12:
                    return cand
            except Exception:
                pass
        return sp.Float(v)
    def clone(self): return Constant(self.value)

class FloatInput(Operation):
    def __init__(self): super().__init__(name="x")
    def eval_backend(self, x, const_map=None): return x
    def to_text(self): return "x"
    def to_sympy(self, xs): return xs
    def clone(self): return FloatInput()

class Unary(Operation):
    def __init__(self, name, o1): super().__init__(name=name, o1=o1)

class Binary(Operation):
    def __init__(self, name, o1, o2): super().__init__(name=name, o1=o1, o2=o2)

# Unarias básicas
class Neg(Unary):
    def __init__(self,o1=None): super().__init__("neg",o1)
    def eval_backend(self,x,c=None): return -(self.o1.eval_backend(x,c))
    def to_text(self): return f"-({self.o1.to_text()})"
    def to_sympy(self, xs): return -self.o1.to_sympy(xs)
    def clone(self): return Neg(self.o1.clone())

class Abs(Unary):
    def __init__(self,o1=None): super().__init__("abs",o1)
    def eval_backend(self,x,c=None): return abs_(self.o1.eval_backend(x,c))
    def to_text(self): return f"abs({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.Abs(self.o1.to_sympy(xs))
    def clone(self): return Abs(self.o1.clone())

class Sin(Unary):
    def __init__(self,o1=None): super().__init__("sin",o1)
    def eval_backend(self,x,c=None): return sin_(self.o1.eval_backend(x,c))
    def to_text(self): return f"sin({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.sin(self.o1.to_sympy(xs))
    def clone(self): return Sin(self.o1.clone())

class Cos(Unary):
    def __init__(self,o1=None): super().__init__("cos",o1)
    def eval_backend(self,x,c=None): return cos_(self.o1.eval_backend(x,c))
    def to_text(self): return f"cos({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.cos(self.o1.to_sympy(xs))
    def clone(self): return Cos(self.o1.clone())

class Tan(Unary):
    def __init__(self,o1=None): super().__init__("tan",o1)
    def eval_backend(self,x,c=None): return tan_(self.o1.eval_backend(x,c))
    def to_text(self): return f"tan({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.tan(self.o1.to_sympy(xs))
    def clone(self): return Tan(self.o1.clone())

class Log(Unary):
    def __init__(self,o1=None): super().__init__("log",o1)
    def eval_backend(self,x,c=None): return log_(self.o1.eval_backend(x,c))
    def to_text(self): return f"log(|{self.o1.to_text()}|+eps)"
    def to_sympy(self, xs): return sp.log(sp.Abs(self.o1.to_sympy(xs)))  # semántica segura: log(|u|)
    def clone(self): return Log(self.o1.clone())

class Sqrt(Unary):
    def __init__(self,o1=None): super().__init__("sqrt",o1)
    def eval_backend(self,x,c=None): return sqrt_(self.o1.eval_backend(x,c))
    def to_text(self): return f"sqrt(|{self.o1.to_text()})"
    def to_sympy(self, xs): return sp.sqrt(sp.Abs(self.o1.to_sympy(xs)))  # semántica segura: sqrt(|u|)
    def clone(self): return Sqrt(self.o1.clone())

class Tanh(Unary):
    def __init__(self,o1=None): super().__init__("tanh",o1)
    def eval_backend(self,x,c=None): return tanh_(self.o1.eval_backend(x,c))
    def to_text(self): return f"tanh({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.tanh(self.o1.to_sympy(xs))
    def clone(self): return Tanh(self.o1.clone())

class Sigmoid(Unary):
    def __init__(self,o1=None): super().__init__("sigmoid",o1)
    def eval_backend(self,x,c=None): return sigmoid_(self.o1.eval_backend(x,c))
    def to_text(self): return f"sigmoid({self.o1.to_text()})"
    def to_sympy(self, xs): return 1/(1 + sp.exp(-self.o1.to_sympy(xs)))
    def clone(self): return Sigmoid(self.o1.clone())

class Asin(Unary):
    def __init__(self,o1=None): super().__init__("asin",o1)
    def eval_backend(self,x,c=None): return asin_(self.o1.eval_backend(x,c))
    def to_text(self): return f"asin({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.asin(self.o1.to_sympy(xs))
    def clone(self): return Asin(self.o1.clone())

class Acos(Unary):
    def __init__(self,o1=None): super().__init__("acos",o1)
    def eval_backend(self,x,c=None): return acos_(self.o1.eval_backend(x,c))
    def to_text(self): return f"acos({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.acos(self.o1.to_sympy(xs))
    def clone(self): return Acos(self.o1.clone())

class Atan(Unary):
    def __init__(self,o1=None): super().__init__("atan",o1)
    def eval_backend(self,x,c=None): return atan_(self.o1.eval_backend(x,c))
    def to_text(self): return f"atan({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.atan(self.o1.to_sympy(xs))
    def clone(self): return Atan(self.o1.clone())

# Especiales / polinomiales
class Erf(Unary):
    def __init__(self,o1=None): super().__init__("erf",o1)
    def eval_backend(self,x,c=None): return erf_(self.o1.eval_backend(x,c))
    def to_text(self): return f"erf({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.erf(self.o1.to_sympy(xs))
    def clone(self): return Erf(self.o1.clone())

class Erfc(Unary):
    def __init__(self,o1=None): super().__init__("erfc",o1)
    def eval_backend(self,x,c=None): return erfc_(self.o1.eval_backend(x,c))
    def to_text(self): return f"erfc({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.erfc(self.o1.to_sympy(xs))
    def clone(self): return Erfc(self.o1.clone())

class J0(Unary):
    def __init__(self,o1=None): super().__init__("j0",o1)
    def eval_backend(self,x,c=None): return j0_(self.o1.eval_backend(x,c))
    def to_text(self): return f"J0({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.besselj(0, self.o1.to_sympy(xs))
    def clone(self): return J0(self.o1.clone())

class J1(Unary):
    def __init__(self,o1=None): super().__init__("j1",o1)
    def eval_backend(self,x,c=None): return j1_(self.o1.eval_backend(x,c))
    def to_text(self): return f"J1({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.besselj(1, self.o1.to_sympy(xs))
    def clone(self): return J1(self.o1.clone())

class Y0(Unary):
    def __init__(self,o1=None): super().__init__("y0",o1)
    def eval_backend(self,x,c=None): return y0_(self.o1.eval_backend(x,c))
    def to_text(self): return f"Y0({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.bessely(0, self.o1.to_sympy(xs))  # el motor recorta x>=1e-6
    def clone(self): return Y0(self.o1.clone())

class Y1(Unary):
    def __init__(self,o1=None): super().__init__("y1",o1)
    def eval_backend(self,x,c=None): return y1_(self.o1.eval_backend(x,c))
    def to_text(self): return f"Y1({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.bessely(1, self.o1.to_sympy(xs))  # el motor recorta x>=1e-6
    def clone(self): return Y1(self.o1.clone())

class Gamma(Unary):
    def __init__(self,o1=None): super().__init__("gamma",o1)
    def eval_backend(self,x,c=None): return gamma_(self.o1.eval_backend(x,c))
    def to_text(self): return f"Gamma({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.Abs(sp.gamma(self.o1.to_sympy(xs)))  # el motor usa exp(lgamma)=|Gamma|
    def clone(self): return Gamma(self.o1.clone())

class ChebyshevT(Unary):
    def __init__(self,o1=None,n=2): super().__init__("chebyshevT",o1); self.n=int(n)
    def eval_backend(self,x,c=None): return chebyshevT_(self.o1.eval_backend(x,c), self.n)
    def to_text(self): return f"T_{self.n}({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.chebyshevt(self.n, self.o1.to_sympy(xs))
    def clone(self): return ChebyshevT(self.o1.clone(), self.n)

class LegendreP(Unary):
    def __init__(self,o1=None,n=2): super().__init__("legendreP",o1); self.n=int(n)
    def eval_backend(self,x,c=None): return legendreP_(self.o1.eval_backend(x,c), self.n)
    def to_text(self): return f"P_{self.n}({self.o1.to_text()})"
    def to_sympy(self, xs): return sp.legendre(self.n, self.o1.to_sympy(xs))
    def clone(self): return LegendreP(self.o1.clone(), self.n)

# Binarias
class Add(Binary):
    def __init__(self,o1=None,o2=None): super().__init__("add",o1,o2)
    def eval_backend(self,x,c=None): return add_(self.o1.eval_backend(x,c), self.o2.eval_backend(x,c))
    def to_text(self): return f"({self.o1.to_text()} + {self.o2.to_text()})"
    def to_sympy(self, xs): return self.o1.to_sympy(xs) + self.o2.to_sympy(xs)
    def clone(self): return Add(self.o1.clone(), self.o2.clone())

class Sub(Binary):
    def __init__(self,o1=None,o2=None): super().__init__("sub",o1,o2)
    def eval_backend(self,x,c=None): return sub_(self.o1.eval_backend(x,c), self.o2.eval_backend(x,c))
    def to_text(self): return f"({self.o1.to_text()} - {self.o2.to_text()})"
    def to_sympy(self, xs): return self.o1.to_sympy(xs) - self.o2.to_sympy(xs)
    def clone(self): return Sub(self.o1.clone(), self.o2.clone())

class Mul(Binary):
    def __init__(self,o1=None,o2=None): super().__init__("mul",o1,o2)
    def eval_backend(self,x,c=None): return mul_(self.o1.eval_backend(x,c), self.o2.eval_backend(x,c))
    def to_text(self): return f"({self.o1.to_text()} * {self.o2.to_text()})"
    def to_sympy(self, xs): return self.o1.to_sympy(xs) * self.o2.to_sympy(xs)
    def clone(self): return Mul(self.o1.clone(), self.o2.clone())

class Div(Binary):
    def __init__(self,o1=None,o2=None): super().__init__("div",o1,o2)
    def eval_backend(self,x,c=None): return div_(self.o1.eval_backend(x,c), self.o2.eval_backend(x,c))
    def to_text(self): return f"({self.o1.to_text()} / {self.o2.to_text()})"
    def to_sympy(self, xs): return self.o1.to_sympy(xs) / self.o2.to_sympy(xs)
    def clone(self): return Div(self.o1.clone(), self.o2.clone())

class Pow(Binary):
    def __init__(self,o1=None,o2=None): super().__init__("pow",o1,o2)
    def eval_backend(self,x,c=None): return pow_(self.o1.eval_backend(x,c), self.o2.eval_backend(x,c))
    def to_text(self): return f"({self.o1.to_text()} ^ {self.o2.to_text()})"
    def to_sympy(self, xs):
        # el motor recorta el exponente a [-5,5]; si es constante, lo reflejamos
        e = self.o2.to_sympy(xs)
        if isinstance(self.o2, Constant):
            ev = min(5.0, max(-5.0, self.o2.value))
            e = sp.Integer(round(ev)) if abs(ev - round(ev)) < 1e-10 else sp.Float(ev)
        return self.o1.to_sympy(xs) ** e
    def clone(self): return Pow(self.o1.clone(), self.o2.clone())

class Min(Binary):
    def __init__(self,o1=None,o2=None): super().__init__("min",o1,o2)
    def eval_backend(self,x,c=None):
        a=self.o1.eval_backend(x,c); b=self.o2.eval_backend(x,c)
        return (torch.minimum(a,b) if BACKEND=="torch" else cp.minimum(a,b))
    def to_text(self): return f"min({self.o1.to_text()}, {self.o2.to_text()})"
    def to_sympy(self, xs): return sp.Min(self.o1.to_sympy(xs), self.o2.to_sympy(xs))
    def clone(self): return Min(self.o1.clone(), self.o2.clone())

class Max(Binary):
    def __init__(self,o1=None,o2=None): super().__init__("max",o1,o2)
    def eval_backend(self,x,c=None):
        a=self.o1.eval_backend(x,c); b=self.o2.eval_backend(x,c)
        return (torch.maximum(a,b) if BACKEND=="torch" else cp.maximum(a,b))
    def to_text(self): return f"max({self.o1.to_text()}, {self.o2.to_text()})"
    def to_sympy(self, xs): return sp.Max(self.o1.to_sympy(xs), self.o2.to_sympy(xs))
    def clone(self): return Max(self.o1.clone(), self.o2.clone())

class Beta(Binary):
    def __init__(self,o1=None,o2=None): super().__init__("beta",o1,o2)
    def eval_backend(self,x,c=None): return beta_(self.o1.eval_backend(x,c), self.o2.eval_backend(x,c))
    def to_text(self): return f"Beta({self.o1.to_text()}, {self.o2.to_text()})"
    def to_sympy(self, xs): return sp.beta(self.o1.to_sympy(xs), self.o2.to_sympy(xs))
    def clone(self): return Beta(self.o1.clone(), self.o2.clone())

class Eml(Binary):
    # Fase 4.1: eml(u,v) = exp(u) - ln(v) (Odrzywolek, arXiv:2603.21852)
    def __init__(self,o1=None,o2=None): super().__init__("eml",o1,o2)
    def eval_backend(self,x,c=None): return eml_(self.o1.eval_backend(x,c), self.o2.eval_backend(x,c))
    def to_text(self): return f"eml({self.o1.to_text()}, {self.o2.to_text()})"
    def to_sympy(self, xs):
        u = self.o1.to_sympy(xs); v = self.o2.to_sympy(xs)
        if EML_SYMPY_EXPAND:
            return sp.exp(u) - sp.log(sp.Abs(v))   # expandida a exp/ln (Fase 4.3, evaluable)
        return sp.Function("eml")(u, v)            # compacta, solo para display
    def clone(self): return Eml(self.o1.clone(), self.o2.clone())




## Catálogo inicial y AOS (auto‑selección de operadores)
Edita `SELECT_UN` y `SELECT_BIN` para fijar el conjunto **inicial**.  
AOS ajustará probabilidades y puede **podar** operadores débiles.


In [ ]:

SELECT_UN = ["sin","cos","log","sqrt","abs","neg"]  # Puedes añadir: "tan","tanh","sigmoid","asin","acos","atan","erf","erfc","gamma","j0","j1","y0","y1","chebyshevT","legendreP"
SELECT_BIN = ["add","sub","mul","div","pow"]        # Puedes añadir: "min","max","beta"
CHEB_DEGREES=[2,3,4]
LEG_DEGREES=[2,3,4]

UNARY_MAP = {"neg":Neg,"abs":Abs,"sin":Sin,"cos":Cos,"tan":Tan,"log":Log,"sqrt":Sqrt,
             "tanh":Tanh,"sigmoid":Sigmoid,"asin":Asin,"acos":Acos,"atan":Atan,
             "erf":Erf,"erfc":Erfc,"gamma":Gamma,"j0":J0,"j1":J1,"y0":Y0,"y1":Y1,
             "chebyshevT":ChebyshevT,"legendreP":LegendreP}
BINARY_MAP={"add":Add,"sub":Sub,"mul":Mul,"div":Div,"pow":Pow,"min":Min,"max":Max,"beta":Beta,"eml":Eml}

def build_catalogs(select_un, select_bin, cheb=CHEB_DEGREES, leg=LEG_DEGREES):
    CAT_UN=[]; CAT_BIN=[]
    for name in select_un:
        if name=="chebyshevT":
            for n in cheb: CAT_UN.append((f"chebyshevT_{n}", lambda o1, n=n: ChebyshevT(o1=o1, n=n)))
        elif name=="legendreP":
            for n in leg:  CAT_UN.append((f"legendreP_{n}", lambda o1, n=n: LegendreP(o1=o1, n=n)))
        else:
            CAT_UN.append((name, lambda o1, cls=UNARY_MAP[name]: cls(o1=o1)))
    for name in select_bin:
        CAT_BIN.append((name, lambda o1,o2,cls=BINARY_MAP[name]: cls(o1=o1, o2=o2)))
    return CAT_UN, CAT_BIN

CAT_UN, CAT_BIN = build_catalogs(SELECT_UN, SELECT_BIN)

if EML_MODE == "operator":
    # Fase 4: eml entra al catálogo inicial y el AOS lo pondera como a cualquier otro
    CAT_UN, CAT_BIN = build_catalogs(SELECT_UN, SELECT_BIN + ["eml"])

print("Unarias candidatas:", [n for n,_ in CAT_UN])
print("Binarias candidatas:", [n for n,_ in CAT_BIN])

# === AOS ===
class OperatorManager:
    def __init__(self, unary_catalog, binary_catalog, tau=1.0, lr=0.3, decay=0.99,
                 prune_every=30, min_keep_un=3, min_keep_bin=3, prune_threshold=0.05):
        self.tau=tau; self.lr=lr; self.decay=decay
        self.prune_every=prune_every; self.min_keep_un=min_keep_un; self.min_keep_bin=min_keep_bin
        self.prune_threshold=prune_threshold
        self.un_list=list(unary_catalog); self.bin_list=list(binary_catalog)
        self.un_scores={n:0.0 for n,_ in self.un_list}; self.bin_scores={n:0.0 for n,_ in self.bin_list}
        self.generation=0

    def softmax_probs(self, scores, names):
        v = np.array([scores[n] for n in names], dtype=np.float64)
        v = v - v.max() if len(v)>0 else v
        p = np.exp(v / max(1e-6, self.tau)) if len(v)>0 else v
        p = p / p.sum() if (len(v)>0 and p.sum()>0) else (np.ones(len(v))/len(v) if len(v)>0 else np.array([]))
        return p

    def sample_unary(self):
        names=[n for n,_ in self.un_list]
        p=self.softmax_probs(self.un_scores, names)
        idx=int(np.random.choice(len(names), p=p))
        return self.un_list[idx], p[idx]

    def sample_binary(self):
        names=[n for n,_ in self.bin_list]
        p=self.softmax_probs(self.bin_scores, names)
        idx=int(np.random.choice(len(names), p=p))
        return self.bin_list[idx], p[idx]

    def update_reward(self, used_ops, reward):
        for name in used_ops:
            if name in self.un_scores: self.un_scores[name] = self.decay*self.un_scores[name] + self.lr*reward
            if name in self.bin_scores: self.bin_scores[name] = self.decay*self.bin_scores[name] + self.lr*reward

    def maybe_prune(self):
        self.generation += 1
        if self.prune_every<=0 or (self.generation % self.prune_every)!=0: return
        # Unarias
        names=[n for n,_ in self.un_list]
        if len(names) > self.min_keep_un:
            p=self.softmax_probs(self.un_scores, names)
            order=list(np.argsort(-p))
            survivors=[self.un_list[i] for i in order if p[i]>=self.prune_threshold]
            if len(survivors) < self.min_keep_un:
                survivors=[self.un_list[i] for i in order[:self.min_keep_un]]
            self.un_list=survivors; self.un_scores={n:self.un_scores[n] for n,_ in self.un_list}
        # Binarias
        names=[n for n,_ in self.bin_list]
        if len(names) > self.min_keep_bin:
            p=self.softmax_probs(self.bin_scores, names)
            order=list(np.argsort(-p))
            survivors=[self.bin_list[i] for i in order if p[i]>=self.prune_threshold]
            if len(survivors) < self.min_keep_bin:
                survivors=[self.bin_list[i] for i in order[:self.min_keep_bin]]
            self.bin_list=survivors; self.bin_scores={n:self.bin_scores[n] for n,_ in self.bin_list}




## Mutación / Crossover + Macros (subárboles) y Hints (semillas)


In [ ]:

TERMINALS=[FloatInput, Constant]
def random_terminal():
    return random.choices([FloatInput, Constant], weights=[3,2], k=1)[0]()

def all_subnodes(node):
    out=[]
    def rec(cur, parent=None, attr=None):
        out.append((cur,parent,attr))
        if cur.o1 is not None: rec(cur.o1, cur, "o1")
        if cur.o2 is not None: rec(cur.o2, cur, "o2")
    rec(node, None, None); return out

def random_node(aos: OperatorManager, max_depth=5):
    if max_depth<=0: return random_terminal(), []
    r=random.random()
    use_bin = len(aos.bin_list)>0
    use_un  = len(aos.un_list)>0
    if use_bin and (not use_un or r<0.5):
        (name, ctor), _ = aos.sample_binary()
        left, ops_l = random_node(aos, max_depth-1)
        right, ops_r = random_node(aos, max_depth-1)
        return ctor(left, right), ops_l+ops_r+[name]
    elif use_un:
        (name, ctor), _ = aos.sample_unary()
        child, ops_c = random_node(aos, max_depth-1)
        return ctor(child), ops_c+[name]
    else:
        return random_terminal(), []

# ==== Macros de usuario (añade las que quieras) ====
def macro_sin_lin():
    # sin(a*x + b)
    return Sin(Add(Mul(Constant(1.0), FloatInput()), Constant(0.0)))

def macro_ax_plus_b():
    return Add(Mul(Constant(1.0), FloatInput()), Constant(0.0))

USER_MACROS = [macro_sin_lin, macro_ax_plus_b]  # edita esta lista

def mutate(aos: OperatorManager, node, max_depth=5, p_point=0.5):
    node=node.clone()
    subs=all_subnodes(node)
    target, parent, attr = random.choice(subs)
    if isinstance(target, Constant) and random.random()<p_point:
        k = random.choice([0.1,0.5,1,2,5])
        new = Constant(target.value + random.uniform(-k,k)); used=[]
    else:
        new, used = random_node(aos, max_depth=max_depth)
    if parent is None: return new, used
    setattr(parent, attr, new); return node, used

def mutate_with_macros(aos: OperatorManager, node, max_depth=5, p_macro=0.2, p_point=0.5):
    if USER_MACROS and random.random() < p_macro:
        node = node.clone()
        subs=all_subnodes(node)
        _, parent, attr = random.choice(subs)
        new = random.choice(USER_MACROS)()
        if parent is None: return new, ['macro']
        setattr(parent, attr, new); return node, ['macro']
    else:
        return mutate(aos, node, max_depth=max_depth, p_point=p_point)

def crossover(aos: OperatorManager, a, b):
    a=a.clone(); b=b.clone()
    sa=all_subnodes(a); sb=all_subnodes(b)
    ta, pa, attra = random.choice(sa)
    tb, pb, attrb = random.choice(sb)
    donated = tb.clone()
    # Fase 3.4: ops del subárbol donado, para poder recompensar al crossover en el AOS
    used = ([n.name for n,_,_ in all_subnodes(donated) if n.name not in ("const","x")]
            if AOS_XOVER_REWARD else [])
    if pa is None: child = donated
    else: setattr(pa, attra, donated); child=a
    return child, used

# ==== Hint (sembrar población con función parecida) ====
def make_hint_tree():
    # Edita esta función para representar tu "pista" g(x)
    # Ejemplo: g(x) = c0 + c1*x + c2*sin(x)
    return Add(Add(Constant(0.0), Mul(Constant(1.0), FloatInput())),
               Mul(Constant(1.0), Sin(FloatInput())))

def inject_hint_population(pop, fraction=0.3, jitter=0.2):
    n = max(1, int(len(pop)*fraction))
    for i in range(n):
        clone = make_hint_tree().clone()
        # jitter en constantes
        for node,_,_ in all_subnodes(clone):
            if isinstance(node, Constant):
                node.value += np.random.uniform(-jitter, jitter)
        pop[i] = clone




## Pérdida con derivadas (opcional) y Fitness
Activa `USE_DERIV_LOSS=True` y define pesos `LAMBDA_*`.  
Si proporcionas `DY` y/o `DDY` con la misma forma que `Y`, se acelerará el hallazgo.


In [ ]:

USE_DERIV_LOSS = False
LAMBDA_Y, LAMBDA_Yp, LAMBDA_Ypp = 1.0, 0.5, 0.2
DY = None   # y'(x)  (to_device más adelante)
DDY = None  # y''(x)

def mse_expr(expr, x_arr, y_arr):
    yhat = expr.eval_backend(x_arr)
    if BACKEND=="torch":
        bad = torch.isnan(yhat).any() or torch.isinf(yhat).any()
        if bad: return 1e12
        return float(torch.mean((yhat - y_arr)**2).item())
    else:
        if cp.isnan(yhat).any() or cp.isinf(yhat).any(): return 1e12
        return float(cp.mean((yhat - y_arr)**2).get())

def mse_expr_with_derivs(expr, x_arr, y_arr, dy_arr=None, ddy_arr=None):
    # y_hat
    yhat = expr.eval_backend(x_arr)
    if BACKEND=="torch":
        mse_y = torch.mean((yhat - y_arr)**2)
        mse_p = torch.tensor(0.0, device=y_arr.device)
        mse_pp= torch.tensor(0.0, device=y_arr.device)

        if (dy_arr is not None) or (ddy_arr is not None):
            x_req = x_arr.clone().detach().requires_grad_(True)
            yhat2 = expr.eval_backend(x_req)
            if dy_arr is not None:
                (dyhat,) = torch.autograd.grad(yhat2, x_req, grad_outputs=torch.ones_like(yhat2),
                                               create_graph=True, retain_graph=True, allow_unused=True)
                if dyhat is None: dyhat = torch.zeros_like(yhat2)
                mse_p = torch.mean((dyhat - dy_arr)**2)
            if ddy_arr is not None:
                (dyhat2,) = torch.autograd.grad(yhat2, x_req, grad_outputs=torch.ones_like(yhat2),
                                                create_graph=True, retain_graph=True, allow_unused=True)
                if dyhat2 is None: dyhat2 = torch.zeros_like(yhat2)
                (ddyhat,) = torch.autograd.grad(dyhat2, x_req, grad_outputs=torch.ones_like(dyhat2),
                                                create_graph=False, retain_graph=False, allow_unused=True)
                if ddyhat is None: ddyhat = torch.zeros_like(yhat2)
                mse_pp = torch.mean((ddyhat - ddy_arr)**2)

        loss = (LAMBDA_Y*mse_y
                + (0 if dy_arr is None else LAMBDA_Yp*mse_p)
                + (0 if ddy_arr is None else LAMBDA_Ypp*mse_pp))
        return float(loss.item())

    else:
        mse_y = cp.mean((yhat - y_arr)**2)
        loss = LAMBDA_Y*mse_y

        if (dy_arr is not None) or (ddy_arr is not None):
            h = cp.asarray(1e-3, dtype=x_arr.dtype)
            fph = expr.eval_backend(x_arr + h)
            fmh = expr.eval_backend(x_arr - h)
            if dy_arr is not None:
                dyhat = (fph - fmh) / (2*h)
                loss = loss + LAMBDA_Yp * cp.mean((dyhat - dy_arr)**2)
            if ddy_arr is not None:
                fpph = expr.eval_backend(x_arr + 2*h)
                fmmh = expr.eval_backend(x_arr - 2*h)
                ddyhat = (fpph - 2*yhat + fmmh) / (h*h)
                loss = loss + LAMBDA_Ypp * cp.mean((ddyhat - ddy_arr)**2)

        return float(loss.get())

_AUTO = object()  # centinela: usa las globales DY/DDY (datos completos)

def fitness(expr, x_arr, y_arr, alpha=1e-3, dy_arr=_AUTO, ddy_arr=_AUTO):
    if USE_DERIV_LOSS:
        d1 = DY if dy_arr is _AUTO else dy_arr
        d2 = DDY if ddy_arr is _AUTO else ddy_arr
        base = mse_expr_with_derivs(expr, x_arr, y_arr, dy_arr=d1, ddy_arr=d2)
    else:
        base = mse_expr(expr, x_arr, y_arr)
    n_nodes = len(all_subnodes(expr))
    return base + alpha*n_nodes


## Evaluación por lotes en GPU + control de bloat (Fase 2)
- `eval_population_batch`: evalúa TODA la población acumulando en un tensor
  `(pop, n_puntos)`, calcula todos los MSE en una sola reducción y sincroniza
  CPU-GPU **una sola vez** por lote (antes: una sincronización por individuo).
- `simplify_tree` / `dedup_population`: control de bloat periódico.


In [ ]:

# Operadores vectorizados por nombre (para el intérprete plano)
_VOPS_UN = {"neg": lambda a: -a, "abs": abs_, "sin": sin_, "cos": cos_, "tan": tan_,
            "log": log_, "sqrt": sqrt_, "tanh": tanh_, "sigmoid": sigmoid_,
            "asin": asin_, "acos": acos_, "atan": atan_, "erf": erf_, "erfc": erfc_,
            "gamma": gamma_, "j0": j0_, "j1": j1_, "y0": y0_, "y1": y1_}
_VOPS_BIN = {"add": add_, "sub": sub_, "mul": mul_, "div": div_, "pow": pow_,
             "min": lambda a,b: torch.minimum(a,b) if BACKEND=="torch" else cp.minimum(a,b),
             "max": lambda a,b: torch.maximum(a,b) if BACKEND=="torch" else cp.maximum(a,b),
             "beta": beta_, "eml": eml_}

def _vop_unary(key):
    name, n = key
    if name == "chebyshevT": return (lambda a, n=n: chebyshevT_(a, n))
    if name == "legendreP":  return (lambda a, n=n: legendreP_(a, n))
    return _VOPS_UN[name]

def _flatten_trees(pop):
    """Aplana una lista de árboles a un DAG por niveles: cada nodo recibe un
    índice global y los operadores se agrupan por (nivel, tipo) para poder
    evaluarse con pocos kernels grandes en vez de miles de kernels pequeños."""
    from collections import defaultdict
    x_rows = []; c_rows = []; c_vals = []; roots = []
    groups = defaultdict(lambda: ([], [], []))   # (nivel, clave_op) -> (out, hijo1, hijo2)
    idx = 0
    def rec(nd):
        nonlocal idx
        if isinstance(nd, FloatInput):
            my = idx; idx += 1; x_rows.append(my); return my, 0
        if isinstance(nd, Constant):
            my = idx; idx += 1; c_rows.append(my); c_vals.append(nd.value); return my, 0
        if nd.o2 is None:
            a, la = rec(nd.o1)
            my = idx; idx += 1
            lvl = la + 1
            g = groups[(lvl, (nd.name, getattr(nd, "n", None)))]
            g[0].append(my); g[1].append(a)
            return my, lvl
        a, la = rec(nd.o1); b, lb = rec(nd.o2)
        my = idx; idx += 1
        lvl = max(la, lb) + 1
        g = groups[(lvl, (nd.name, None))]
        g[0].append(my); g[1].append(a); g[2].append(b)
        return my, lvl
    for t in pop:
        r, _ = rec(t); roots.append(r)
    return idx, x_rows, c_rows, c_vals, roots, groups

def eval_trees_matrix(pop, x_arr):
    """Evalúa todos los árboles y devuelve el tensor (pop, n_puntos) SIN
    sincronizar: intérprete plano por niveles con kernels agrupados por operador."""
    n_tot, x_rows, c_rows, c_vals, roots, groups = _flatten_trees(pop)
    dev = x_arr.device
    x_flat = x_arr.reshape(-1)
    B = torch.empty((n_tot, x_flat.shape[0]), dtype=x_arr.dtype, device=dev)
    if x_rows:
        B[torch.as_tensor(x_rows, device=dev)] = x_flat
    if c_rows:
        vals = torch.as_tensor(np.asarray(c_vals, dtype=np.float64), dtype=x_arr.dtype, device=dev)
        B[torch.as_tensor(c_rows, device=dev)] = vals[:, None]
    for (lvl, key) in sorted(groups.keys()):
        out_i, a_i, b_i = groups[(lvl, key)]
        oi = torch.as_tensor(out_i, device=dev)
        A = B[torch.as_tensor(a_i, device=dev)]
        if b_i:
            R = _VOPS_BIN[key[0]](A, B[torch.as_tensor(b_i, device=dev)])
        else:
            R = _vop_unary(key)(A)
        B[oi] = R
    return B[torch.as_tensor(roots, device=dev)]

def eval_population_batch(pop, x_arr, y_arr, alpha=1e-3, dy_arr=None, ddy_arr=None):
    """Evalúa una lista de expresiones por lotes en GPU y devuelve el vector de
    fitness (np.ndarray). Una única transferencia GPU->CPU por llamada."""
    if BACKEND != "torch":
        # CuPy: ruta clásica individuo a individuo
        return np.array([fitness(e, x_arr, y_arr, alpha=alpha, dy_arr=dy_arr, ddy_arr=ddy_arr)
                         for e in pop], dtype=np.float64)
    use_derivs = USE_DERIV_LOSS and ((dy_arr is not None) or (ddy_arr is not None))
    if not use_derivs:
        with torch.no_grad():
            Yh = eval_trees_matrix(pop, x_arr)               # (pop, n_puntos)
            yt = y_arr.reshape(1, -1)
            loss = torch.mean((Yh - yt)**2, dim=1)           # todos los MSE en una reducción
            # Misma semántica que mse_expr: yhat con NaN/Inf -> 1e12; un MSE que
            # desborda con yhat finito se queda en inf (igual que la ruta clásica)
            bad = ~torch.isfinite(Yh).all(dim=1)
            loss = torch.where(bad, torch.full_like(loss, 1e12), loss)
    else:
        # Con pérdida de derivadas: autograd por individuo, sin sincronizar hasta el final
        losses = []
        for e in pop:
            x_req = x_arr.clone().detach().requires_grad_(True)
            yh = e.eval_backend(x_req)
            l = LAMBDA_Y*torch.mean((yh - y_arr)**2)
            (dyh,) = torch.autograd.grad(yh, x_req, grad_outputs=torch.ones_like(yh),
                                         create_graph=(ddy_arr is not None), retain_graph=True,
                                         allow_unused=True)
            if dyh is None: dyh = torch.zeros_like(yh)
            if dy_arr is not None:
                l = l + LAMBDA_Yp*torch.mean((dyh - dy_arr)**2)
            if ddy_arr is not None:
                (ddyh,) = torch.autograd.grad(dyh, x_req, grad_outputs=torch.ones_like(dyh),
                                              allow_unused=True)
                if ddyh is None: ddyh = torch.zeros_like(yh)
                l = l + LAMBDA_Ypp*torch.mean((ddyh - ddy_arr)**2)
            losses.append(l.detach())
        loss = torch.stack(losses)
        loss = torch.where(~torch.isfinite(loss), torch.full_like(loss, 1e12), loss)
    fits = loss.detach().cpu().numpy().astype(np.float64)    # ÚNICA sincronización CPU-GPU
    n_nodes = np.array([len(all_subnodes(e)) for e in pop], dtype=np.float64)
    return fits + alpha*n_nodes

# --- Simplificación algebraica ligera (Fase 2.5) ---
def _const_value(node):
    # Valor numérico de un subárbol sin x (respeta las guardas numéricas del motor)
    xtmp = to_device(np.zeros((1,1), dtype=np.float32))
    return float(to_numpy(node.eval_backend(xtmp)).reshape(-1)[0])

def _is_const(node, val, tol=1e-12):
    return isinstance(node, Constant) and abs(node.value - val) <= tol

def _has_no_x(node):
    if isinstance(node, FloatInput): return False
    if node.o1 is not None and not _has_no_x(node.o1): return False
    if node.o2 is not None and not _has_no_x(node.o2): return False
    return True

def simplify_tree(node):
    """Simplificación ligera: plegado de constantes (sin(0.5), 2*3, ...) e
    identidades x*1, x+0, x-0, x/1, x^1, x^0, u*0, -(-u). Devuelve árbol nuevo."""
    node = node.clone()
    def rec(cur):
        if cur.o1 is not None: cur.o1 = rec(cur.o1)
        if cur.o2 is not None: cur.o2 = rec(cur.o2)
        # Plegado de constantes: cualquier subárbol sin x se colapsa a Constant
        if not isinstance(cur, (Constant, FloatInput)) and _has_no_x(cur):
            try:
                v = _const_value(cur)
                if np.isfinite(v) and abs(v) < 1e12:
                    return Constant(v)
            except Exception:
                pass
        name = cur.name
        if name == "add":
            if _is_const(cur.o1, 0.0): return cur.o2
            if _is_const(cur.o2, 0.0): return cur.o1
        elif name == "sub":
            if _is_const(cur.o2, 0.0): return cur.o1
        elif name == "mul":
            if _is_const(cur.o1, 1.0): return cur.o2
            if _is_const(cur.o2, 1.0): return cur.o1
            if _is_const(cur.o1, 0.0) or _is_const(cur.o2, 0.0): return Constant(0.0)
        elif name == "div":
            if _is_const(cur.o2, 1.0): return cur.o1
        elif name == "pow":
            if _is_const(cur.o2, 1.0): return cur.o1
            if _is_const(cur.o2, 0.0): return Constant(1.0)
        elif name == "neg" and cur.o1.name == "neg":
            return cur.o1.o1
        return cur
    return rec(node)

# --- Orden NSGA-II (Fase 3.1): frente de Pareto precisión vs complejidad ---
def _nsga2_order(mse, size):
    """Devuelve (orden, rango, crowding) para los objetivos (mse, tamaño):
    sort no dominado rápido O(P^2) + distancia de crowding."""
    mse = np.asarray(mse, dtype=np.float64); size = np.asarray(size, dtype=np.float64)
    P = len(mse)
    # dom[i,j] = True si i domina a j
    dom = ((mse[:,None] <= mse[None,:]) & (size[:,None] <= size[None,:]) &
           ((mse[:,None] < mse[None,:]) | (size[:,None] < size[None,:])))
    n_dom = dom.sum(axis=0).astype(np.int64)
    rank = np.full(P, -1, dtype=np.int64)
    remaining = np.ones(P, dtype=bool)
    fronts = []; fr = 0
    while remaining.any():
        cur = remaining & (n_dom <= 0)
        if not cur.any():
            cur = remaining.copy()   # salvaguarda ante empates degenerados
        idxs = np.where(cur)[0]
        fronts.append(idxs); rank[idxs] = fr
        remaining[idxs] = False
        n_dom = n_dom - dom[idxs].sum(axis=0)
        fr += 1
    crowd = np.zeros(P)
    for fidx in fronts:
        if len(fidx) <= 2:
            crowd[fidx] = np.inf; continue
        for obj in (mse, size):
            o = obj[fidx]; s = np.argsort(o)
            rng = float(o[s[-1]] - o[s[0]]) or 1.0
            crowd[fidx[s[0]]] = crowd[fidx[s[-1]]] = np.inf
            crowd[fidx[s[1:-1]]] += (o[s[2:]] - o[s[:-2]])/rng
    order = sorted(range(P), key=lambda i: (rank[i], -crowd[i]))
    return order, rank, crowd

# --- Deduplicación semántica (Fase 2.5) ---
def dedup_population(pop, aos, max_depth, lo=-3.0, hi=3.0, n_probe=16):
    """Firma semántica = valores de la expresión en n_probe puntos (evaluados en
    GPU por lotes). Los duplicados funcionales se sustituyen por árboles frescos."""
    x_probe = to_device(np.linspace(lo, hi, n_probe).reshape(-1,1).astype(np.float32))
    if BACKEND == "torch":
        with torch.no_grad():
            M = to_numpy(eval_trees_matrix(pop, x_probe))
    else:
        M = np.stack([to_numpy(e.eval_backend(x_probe)).reshape(-1) for e in pop])
    M = np.nan_to_num(M, nan=1e30, posinf=1e30, neginf=-1e30)
    seen = set(); out = list(pop); n_rep = 0
    for i, row in enumerate(M):
        sig = np.round(row, 4).tobytes()
        if sig in seen:
            out[i], _ = random_node(aos, max_depth=max_depth)
            n_rep += 1
        else:
            seen.add(sig)
    return out, n_rep


## Evolución con aceleradores
- **Currículum**: fases de generaciones con diferente `max_depth` y catálogos activos.
- **Submuestreo**: usa fracciones crecientes de los datos para evaluar.
- **Priors**: impulso inicial de probabilidades para operadores coherentes.
- **Memetic**: hill-climb local sobre el mejor individuo de cada generación.


In [ ]:

# === Hall of Fame (Fase 1.2) ===
# La evolución evalúa con submuestreo; el mejor histórico podía perderse porque
# solo se reevaluaba la población FINAL con los datos completos. Este archivo
# guarda los k mejores de TODA la corrida, siempre evaluados con datos completos.
class HallOfFame:
    def __init__(self, k=10):
        self.k = int(k)
        self.items = []   # lista de (fitness_full, expr) ordenada ascendente

    def update(self, expr, fit_full):
        if not np.isfinite(fit_full): return
        key = expr.to_text()
        for i, (f, e) in enumerate(self.items):
            if e.to_text() == key:
                if fit_full < f:
                    self.items[i] = (fit_full, expr.clone())
                    self.items.sort(key=lambda t: t[0])
                return
        self.items.append((fit_full, expr.clone()))
        self.items.sort(key=lambda t: t[0])
        del self.items[self.k:]

    def best(self):
        return self.items[0] if self.items else (None, None)


def evolve(X, Y, cat_un, cat_bin, pop_size=160, generations=180, max_depth=5,
           elite=10, p_mut=0.6, p_xover=0.3, alpha=1e-3, verbose=True,
           aos_params=None,
           # Aceleradores:
           use_macros=True, p_macro=0.2,
           use_hint=True, hint_fraction=0.3, hint_jitter=0.2,
           # Priors (dict nombre->boost)
           unary_priors=None, binary_priors=None,
           # Submuestreo (lista de (gen_limite, fraccion), orden creciente)
           sample_schedule=None,
           # Currículum de complejidad: lista de dict por fase
           #   cada fase: {"until":g, "depth":d, "un": [...], "bin":[...] } (un/bin opcionales)
           curriculum=None,
           # Memetic
           hill_climb_steps=3,
           # Hall of Fame: nº de mejores históricos archivados (datos completos)
           hof_k=10,
           # --- Fase 3 ---
           selection=None,          # None => flag global SELECTION ("alpha" | "nsga2")
           val_fraction=0.0,        # >0: split train/validación y selección final por validación
           early_stop_patience=0,   # >0: corta la corrida si el mejor (full) no mejora en N gens
           restart_patience=0,      # >0: reinicio con diversidad fresca cada N gens sin mejora
           stagnation_tol=1e-10,    # mejora mínima para resetear el contador de estancamiento
           init_pop=None,           # población inicial (islas / continuación de corridas)
           init_aos=None,           # OperatorManager inicial (islas)
           gen_offset=0):           # desplazamiento de generación para curriculum/submuestreo (islas)
    global DY, DDY
    X = np.asarray(X); Y = np.asarray(Y)

    # Split train/validación (Fase 3.5): la evolución solo ve train
    X_val = Y_val = val_idx = None
    if val_fraction and val_fraction > 0.0:
        if USE_DERIV_LOSS and (DY is not None or DDY is not None):
            assert isinstance(DY, (np.ndarray, type(None))) and isinstance(DDY, (np.ndarray, type(None))), \
                "val_fraction>0 requiere DY/DDY como np.ndarray (aún sin convertir a tensor)"
        perm = np.random.permutation(len(X))
        n_val = max(4, int(len(X)*val_fraction))
        val_idx, tr_idx = perm[:n_val], perm[n_val:]
        X_val, Y_val = X[val_idx], Y[val_idx]
        X, Y = X[tr_idx], Y[tr_idx]

    # Datos completos (train) a dispositivo
    X_full = to_device(X.reshape(-1,1))
    Y_full = to_device(Y.reshape(-1,1))

    # Escala de Y para normalizar recompensas AOS (Fase 3.4)
    y_scale = (float(np.var(np.asarray(Y, dtype=np.float64))) + 1e-12) if AOS_REWARD_NORM else 1.0

    # Selección: "alpha" clásica o frente de Pareto NSGA-II (Fase 3.1)
    sel = selection if selection is not None else SELECTION

    # Derivadas (si existen)
    if val_idx is not None:
        if DY is not None and isinstance(DY, np.ndarray): DY = DY[tr_idx]
        if DDY is not None and isinstance(DDY, np.ndarray): DDY = DDY[tr_idx]
    if DY is not None and not (BACKEND=="torch" and isinstance(DY, (torch.Tensor,))) and not (BACKEND=="cupy" and isinstance(DY, (cp.ndarray,))):
        DY = to_device(DY.reshape(-1,1))
    if DDY is not None and not (BACKEND=="torch" and isinstance(DDY, (torch.Tensor,))) and not (BACKEND=="cupy" and isinstance(DDY, (cp.ndarray,))):
        DDY = to_device(DDY.reshape(-1,1))

    if init_aos is not None:
        aos = init_aos    # continuación (islas): conserva scores y catálogo podado
    else:
        if aos_params is None: aos_params = {}
        aos = OperatorManager(cat_un, cat_bin, **aos_params)
        # Priorización inicial
        if unary_priors:
            for k,v in unary_priors.items():
                if k in aos.un_scores: aos.un_scores[k] += float(v)
        if binary_priors:
            for k,v in binary_priors.items():
                if k in aos.bin_scores: aos.bin_scores[k] += float(v)

    # Población inicial (o continuación desde init_pop)
    if init_pop is not None:
        pop = [e.clone() for e in init_pop[:pop_size]]
        while len(pop) < pop_size:
            pop.append(random_node(aos, max_depth=max_depth)[0])
    else:
        pop=[]
        for _ in range(pop_size):
            ind, used = random_node(aos, max_depth=max_depth)
            pop.append(ind)
        if use_hint:
            inject_hint_population(pop, fraction=hint_fraction, jitter=hint_jitter)

    # Hall of Fame y fitness sobre datos completos
    hof = HallOfFame(k=hof_k)
    def fit_full(e):
        return fitness(e, X_full, Y_full, alpha=alpha)

    # Helpers de evaluación con submuestreo (también recorta DY/DDY al subconjunto)
    def pick_subset(gen):
        if not sample_schedule: return X_full, Y_full, DY, DDY
        # Encuentra última fracción aplicable
        frac = 1.0
        for g_lim, f in sample_schedule:
            if gen <= g_lim: frac = f; break
            frac = f
        n = max(8, int(len(X)*frac))
        idx = np.random.choice(len(X), size=n, replace=False)
        dy_s  = DY[idx]  if DY  is not None else None
        ddy_s = DDY[idx] if DDY is not None else None
        return X_full[idx], Y_full[idx], dy_s, ddy_s

    # Curriculum: función para reconfigurar catálogos y profundidad
    def apply_curriculum(gen):
        nonlocal max_depth, aos
        if not curriculum: return
        for phase in curriculum:
            if gen <= phase.get("until", generations):
                max_depth = phase.get("depth", max_depth)
                un_names = phase.get("un", None)
                bin_names= phase.get("bin", None)
                if un_names is not None or bin_names is not None:
                    new_un, new_bin = build_catalogs(un_names if un_names is not None else [n for n,_ in aos.un_list],
                                                     bin_names if bin_names is not None else [n for n,_ in aos.bin_list])
                    # Mantén scores previos cuando posible
                    old_un_scores = dict(aos.un_scores); old_bin_scores=dict(aos.bin_scores)
                    aos.un_list = list(new_un); aos.bin_list=list(new_bin)
                    aos.un_scores={n:old_un_scores.get(n,0.0) for n,_ in aos.un_list}
                    aos.bin_scores={n:old_bin_scores.get(n,0.0) for n,_ in aos.bin_list}
                break

    def evaluate(pop, fit_fn):
        fits=[fit_fn(ind) for ind in pop]
        order=np.argsort(fits); return [pop[i] for i in order], [fits[i] for i in order]

    x_lo, x_hi = float(np.min(X)), float(np.max(X))
    use_batch = BATCH_EVAL and BACKEND == "torch"

    # Estancamiento (Fase 3.3)
    best_track = np.inf; stagn = 0
    rank_key = {}   # claves (frente, -crowding) por individuo cuando sel=="nsga2"

    best_hist=[]
    for g in range(generations):
        apply_curriculum(g + gen_offset)
        x_arr, y_arr, dy_s, ddy_s = pick_subset(g + gen_offset)

        # Control de bloat periódico (Fase 2.5)
        if SIMPLIFY_EVERY and g and g % SIMPLIFY_EVERY == 0:
            pop = [simplify_tree(e) for e in pop]
        if DEDUP_EVERY and g and g % DEDUP_EVERY == 0:
            pop, _ = dedup_population(pop, aos, max_depth, lo=x_lo, hi=x_hi)

        # --- Evaluación de la población ---
        fit_cache = {}  # caché por individuo y generación (Fase 2.1)
        def fit_sub(e):
            key = id(e)
            if FITNESS_CACHE and key in fit_cache: return fit_cache[key]
            v = fitness(e, x_arr, y_arr, alpha=alpha, dy_arr=dy_s, ddy_arr=ddy_s)
            if FITNESS_CACHE: fit_cache[key] = v
            return v

        if use_batch:
            farr = eval_population_batch(pop, x_arr, y_arr, alpha=alpha, dy_arr=dy_s, ddy_arr=ddy_s)
        else:
            farr = np.array([fit_sub(e) for e in pop], dtype=np.float64)
        if sel == "nsga2":
            # NSGA-II: ordena por (frente no dominado, -crowding) sobre (mse, tamaño)
            nn_arr = np.array([len(all_subnodes(e)) for e in pop], dtype=np.float64)
            order, _rk, _cw = _nsga2_order(farr - alpha*nn_arr, nn_arr)
            rank_key = {id(pop[i]): (int(_rk[i]), -float(_cw[i])) for i in range(len(pop))}
        else:
            order = np.argsort(farr)
        pop  = [pop[i] for i in order]
        fits = [float(farr[i]) for i in order]
        for e, f in zip(pop, fits): fit_cache[id(e)] = f

        best, best_fit = pop[0], fits[0]
        best_hist.append(best_fit)
        hof.update(best, fit_full(best))  # archiva el mejor de la generación (datos completos)
        if verbose and (g%10==0 or g==generations-1):
            print(f"Gen {g:3d}  best={best_fit:.6g}   expr={best.to_text()}")
        aos.maybe_prune()

        # --- Estancamiento: early stopping y reinicio con diversidad (Fase 3.3) ---
        best_full_now = hof.items[0][0] if hof.items else np.inf
        if best_full_now < best_track - stagnation_tol:
            best_track = best_full_now; stagn = 0
        else:
            stagn += 1
        if early_stop_patience and stagn >= early_stop_patience:
            if verbose: print(f"Gen {g}: early stopping ({stagn} generaciones sin mejora)")
            break
        if restart_patience and stagn > 0 and stagn % restart_patience == 0:
            # Inyección de diversidad fresca preservando el elite
            if verbose: print(f"Gen {g}: estancamiento ({stagn} gens) -> reinicio parcial de la población")
            pop = pop[:elite] + [random_node(aos, max_depth=max_depth)[0]
                                 for _ in range(pop_size - elite)]
            for _scores in (aos.un_scores, aos.bin_scores):
                for _k in _scores: _scores[_k] = 0.0
            continue

        # Selección torneo (usa la caché: la población ya está evaluada;
        # con NSGA-II compara por (frente, -crowding))
        def tournament(k=3):
            pool = pop[:50] + pop
            cand = random.sample(pool, k=min(k, len(pool)))
            if sel == "nsga2":
                return min(cand, key=lambda c: rank_key.get(id(c), (10**9, 0.0)))
            vals = [fit_sub(c) for c in cand]
            return cand[int(np.argmin(vals))]

        new_pop = pop[:elite]
        k_hc = max(0, int(hill_climb_steps))

        if use_batch:
            # Memetic secuencial (idéntico a la ruta clásica): pocas evaluaciones
            # individuales por generación, coste despreciable frente al lote
            cur = best; cur_fit = best_fit
            for _ in range(k_hc):
                child, _u = (mutate_with_macros(aos, cur, max_depth=max_depth) if use_macros
                             else mutate(aos, cur, max_depth=max_depth))
                child_fit = fitness(child, x_arr, y_arr, alpha=alpha, dy_arr=dy_s, ddy_arr=ddy_s)
                if child_fit < cur_fit:
                    cur, cur_fit = child, child_fit
            new_pop[0] = cur
            if cur is not best:
                hof.update(cur, fit_full(cur))

            # Descendencia por SUB-LOTES: los torneos solo necesitan los fitness
            # de pop (en caché), y las recompensas AOS se aplican tras cada
            # sub-lote — la selección adaptativa de operadores necesita
            # recompensas casi inmediatas para sesgar la generación en curso
            chunk = max(1, int(AOS_UPDATE_CHUNK))
            if chunk == 1:
                # Recompensa inmediata por hijo (semántica exacta del original);
                # para 1 árbol la evaluación recursiva es más barata que aplanar
                while len(new_pop) < pop_size:
                    parent = tournament()
                    base_fit = fit_sub(parent)
                    r = random.random()
                    if r < p_mut:
                        child, used = (mutate_with_macros(aos, parent, max_depth=max_depth) if use_macros
                                       else mutate(aos, parent, max_depth=max_depth))
                        child_fit = fitness(child, x_arr, y_arr, alpha=alpha, dy_arr=dy_s, ddy_arr=ddy_s)
                        if used: aos.update_reward(used_ops=used, reward=max(0.0, base_fit - child_fit)/y_scale)
                    elif r < p_mut + p_xover:
                        p2 = tournament()
                        child, used = crossover(aos, parent, p2)
                        child_fit = fitness(child, x_arr, y_arr, alpha=alpha, dy_arr=dy_s, ddy_arr=ddy_s)
                        # Fase 3.4: el crossover también recibe recompensa AOS
                        if used: aos.update_reward(used_ops=used, reward=max(0.0, base_fit - child_fit)/y_scale)
                    else:
                        child, used = random_node(aos, max_depth=max_depth)
                        child_fit = fitness(child, x_arr, y_arr, alpha=alpha, dy_arr=dy_s, ddy_arr=ddy_s)
                        if used: aos.update_reward(used_ops=used, reward=max(0.0, best_fit - child_fit)/y_scale)
                    new_pop.append(child)
                pop = new_pop
                continue
            while len(new_pop) < pop_size:
                children = []; meta = []
                while len(children) < chunk and len(new_pop) + len(children) < pop_size:
                    parent = tournament()
                    r = random.random()
                    if r < p_mut:
                        child, used = (mutate_with_macros(aos, parent, max_depth=max_depth) if use_macros
                                       else mutate(aos, parent, max_depth=max_depth))
                        meta.append(("mut", used, fit_sub(parent)))
                    elif r < p_mut + p_xover:
                        p2 = tournament()
                        child, used = crossover(aos, parent, p2)
                        meta.append(("xov", used, fit_sub(parent)))
                    else:
                        child, used = random_node(aos, max_depth=max_depth)
                        meta.append(("rand", used, best_fit))
                    children.append(child)

                # una evaluación por lotes del sub-lote (una sincronización)
                bf = eval_population_batch(children, x_arr, y_arr, alpha=alpha, dy_arr=dy_s, ddy_arr=ddy_s)
                for (kind, used, base), f_child in zip(meta, bf):
                    if used:  # "mut", "rand" y también "xov" (Fase 3.4)
                        aos.update_reward(used_ops=used, reward=max(0.0, base - float(f_child))/y_scale)
                new_pop.extend(children)
        else:
            # --- Ruta clásica (BATCH_EVAL=False): comportamiento original ---
            # Memetic: pequeñas mutaciones sobre el mejor
            cur = best
            for _ in range(k_hc):
                child, used = mutate_with_macros(aos, cur, max_depth=max_depth) if use_macros else mutate(aos, cur, max_depth=max_depth)
                if fit_sub(child) < fit_sub(cur):
                    cur = child
            new_pop[0] = cur  # reemplaza élite 0 por versión mejorada si aplica
            if cur is not best:
                hof.update(cur, fit_full(cur))

            while len(new_pop) < pop_size:
                parent = tournament()
                base_fit = fit_sub(parent)

                r=random.random()
                if r < p_mut:
                    child, used = (mutate_with_macros(aos, parent, max_depth=max_depth) if use_macros
                                   else mutate(aos, parent, max_depth=max_depth))
                    child_fit = fit_sub(child)
                    reward = max(0.0, base_fit - child_fit)/y_scale
                    if used: aos.update_reward(used_ops=used, reward=reward)
                elif r < p_mut + p_xover:
                    p2 = tournament()
                    child, used = crossover(aos, parent, p2)
                    child_fit = fit_sub(child)
                    # Fase 3.4: el crossover también recibe recompensa AOS
                    reward = max(0.0, base_fit - child_fit)/y_scale
                    if used: aos.update_reward(used_ops=used, reward=reward)
                else:
                    child, used = random_node(aos, max_depth=max_depth)
                    child_fit = fit_sub(child)
                    reward = max(0.0, best_fit - child_fit)/y_scale
                    if used: aos.update_reward(used_ops=used, reward=reward)

                new_pop.append(child)
        pop = new_pop

    # evaluación final con todos los datos (train) y actualización del Hall of Fame
    if use_batch:
        farr = eval_population_batch(pop, X_full, Y_full, alpha=alpha, dy_arr=DY, ddy_arr=DDY)
        order = np.argsort(farr)
        pop  = [pop[i] for i in order]
        fits = [float(farr[i]) for i in order]
    else:
        pop, fits = evaluate(pop, fit_full)
    for e, f in zip(pop[:hof_k], fits[:hof_k]):
        hof.update(e, f)

    # Frente de Pareto precisión-complejidad de la población final (Fase 3.1)
    if sel == "nsga2":
        nn_f = np.array([len(all_subnodes(e)) for e in pop], dtype=np.float64)
        mse_f = np.array(fits) - alpha*nn_f
        _o, rk_f, _c = _nsga2_order(mse_f, nn_f)
        seen_tx = set(); pareto = []
        for i in range(len(pop)):
            if rk_f[i] == 0:
                tx = pop[i].to_text()
                if tx not in seen_tx:
                    seen_tx.add(tx); pareto.append((float(mse_f[i]), int(nn_f[i]), pop[i].clone()))
        hof.pareto = sorted(pareto, key=lambda t: t[0])
    else:
        hof.pareto = None

    # Selección del mejor por validación (Fase 3.5)
    hof.val_idx = val_idx
    hof.val_mse = None
    if X_val is not None and hof.items:
        xv = to_device(X_val.reshape(-1,1)); yv = to_device(Y_val.reshape(-1,1))
        scored = sorted(((mse_expr(e, xv, yv), f, e) for f, e in hof.items), key=lambda t: t[0])
        hof.val_mse = {e.to_text(): float(vm) for vm, f, e in scored}
        hof.items = [(f, e) for vm, f, e in scored]  # reordenado: el mejor por validación primero

    hof.final_pop = pop   # estado final (islas / continuación)
    best_fit_full, best_full = hof.items[0]
    return best_full.clone(), best_fit_full, best_hist, aos, hof


def evolve_islands(X, Y, cat_un, cat_bin, n_islands=4, migrate_every=50, migrants=3,
                   generations=1000, pop_size=160, verbose=True, hof_k=10,
                   val_fraction=0.0, **kwargs):
    """Fase 3.6: varias poblaciones (islas) evolucionando en la misma GPU con
    migración periódica en anillo: cada `migrate_every` generaciones los
    `migrants` mejores de la isla i-1 sustituyen a los peores de la isla i.
    Devuelve (best, fit, hist, aos_isla0, hof_global) como evolve()."""
    X = np.asarray(X); Y = np.asarray(Y)
    # split de validación ÚNICO para todas las islas (no por bloque)
    X_val = Y_val = None
    if val_fraction and val_fraction > 0.0:
        perm = np.random.permutation(len(X))
        n_val = max(4, int(len(X)*val_fraction))
        X_val, Y_val = X[perm[:n_val]], Y[perm[:n_val]]
        X, Y = X[perm[n_val:]], Y[perm[n_val:]]

    pops = [None]*n_islands; aoss = [None]*n_islands
    hof_g = HallOfFame(k=hof_k); hist = []
    done = 0
    while done < generations:
        block = min(migrate_every, generations - done)
        for i in range(n_islands):
            _b, _f, _h, aos_i, hof_i = evolve(
                X, Y, cat_un, cat_bin, pop_size=pop_size, generations=block,
                verbose=False, hof_k=hof_k, init_pop=pops[i], init_aos=aoss[i],
                gen_offset=done, **kwargs)
            pops[i] = hof_i.final_pop; aoss[i] = aos_i
            for f, e in hof_i.items:
                hof_g.update(e, f)
        # migración en anillo (las poblaciones finales vienen ordenadas por fitness)
        for i in range(n_islands):
            src = pops[(i-1) % n_islands]; dst = pops[i]
            for k in range(min(migrants, len(src), len(dst))):
                dst[-(k+1)] = src[k].clone()
        done += block
        hist.append(hof_g.items[0][0] if hof_g.items else np.inf)
        if verbose:
            print(f"[islas] gen {done:4d}  best_full={hist[-1]:.6g}")

    # selección final por validación si se pidió
    if X_val is not None and hof_g.items:
        xv = to_device(X_val.reshape(-1,1)); yv = to_device(Y_val.reshape(-1,1))
        scored = sorted(((mse_expr(e, xv, yv), f, e) for f, e in hof_g.items), key=lambda t: t[0])
        hof_g.val_mse = {e.to_text(): float(vm) for vm, f, e in scored}
        hof_g.items = [(f, e) for vm, f, e in scored]
    f0, e0 = hof_g.items[0]
    return e0.clone(), f0, hist, aoss[0], hof_g




## Fine‑tuning de constantes
- **Torch**: Adam sobre constantes (autograd).
- **CuPy**: si hay SciPy, Nelder–Mead (CPU) evaluando la loss en GPU.


In [ ]:

def collect_constants(expr):
    return [n for n,_,_ in all_subnodes(expr) if isinstance(n, Constant)]

def _derivs_to_device():
    # Asegura que DY/DDY estén en el dispositivo con la forma correcta
    global DY, DDY
    if DY is not None and isinstance(DY, np.ndarray):
        DY = to_device(DY.reshape(-1,1))
    if DDY is not None and isinstance(DDY, np.ndarray):
        DDY = to_device(DDY.reshape(-1,1))
    return DY, DDY

def _ft_loss_torch(expr, params, x_t, y_t, dy_t=None, ddy_t=None):
    """Pérdida de fine-tuning (torch), idéntica a la de la evolución (Fase 1.4):
    LAMBDA_Y*MSE(y) + LAMBDA_Yp*MSE(y') + LAMBDA_Ypp*MSE(y'') si hay derivadas."""
    yhat = expr.eval_backend(x_t, params)
    loss = LAMBDA_Y*torch.mean((yhat - y_t)**2)
    if (dy_t is not None) or (ddy_t is not None):
        # Derivadas de la expresión respecto a x vía autograd (create_graph
        # para que el gradiente fluya hasta las constantes)
        x_req = x_t.clone().detach().requires_grad_(True)
        yhat2 = expr.eval_backend(x_req, params)
        (dyhat,) = torch.autograd.grad(yhat2, x_req, grad_outputs=torch.ones_like(yhat2),
                                       create_graph=True, retain_graph=True, allow_unused=True)
        if dyhat is None: dyhat = torch.zeros_like(yhat2)
        if dy_t is not None:
            loss = loss + LAMBDA_Yp*torch.mean((dyhat - dy_t)**2)
        if ddy_t is not None:
            (ddyhat,) = torch.autograd.grad(dyhat, x_req, grad_outputs=torch.ones_like(dyhat),
                                            create_graph=True, retain_graph=True, allow_unused=True)
            if ddyhat is None: ddyhat = torch.zeros_like(yhat2)
            loss = loss + LAMBDA_Ypp*torch.mean((ddyhat - ddy_t)**2)
    return loss

def fine_tune_constants(expr, X, Y, steps=300, lr=5e-3, verbose=True):
    """Ajusta las constantes minimizando la MISMA pérdida que la evolución (Fase 1.4):
    LAMBDA_Y*MSE(y) + LAMBDA_Yp*MSE(y') + LAMBDA_Ypp*MSE(y'') si USE_DERIV_LOSS=True."""
    nodes = collect_constants(expr)
    if not nodes:
        if verbose: print("No hay constantes que ajustar.")
        return expr

    if BACKEND=="torch":
        x_t = to_device(np.asarray(X).reshape(-1,1)); y_t = to_device(np.asarray(Y).reshape(-1,1))
        dy_t, ddy_t = _derivs_to_device() if USE_DERIV_LOSS else (None, None)
        params = {c._id: torch.tensor([c.value], dtype=x_t.dtype, device=x_t.device, requires_grad=True) for c in nodes}
        opt = torch.optim.Adam(params.values(), lr=lr)
        for i in range(1, steps+1):
            opt.zero_grad(set_to_none=True)
            loss = _ft_loss_torch(expr, params, x_t, y_t, dy_t, ddy_t)
            loss.backward(); opt.step()
            if verbose and (i%50==0 or i==1): print(f"FT step {i:4d}  loss={loss.item():.6g}")
        for c in nodes:
            v = float(params[c._id].detach().cpu().item())
            if np.isfinite(v): c.value = v   # si Adam divergió (NaN/Inf), conserva el valor previo
        return expr
    else:
        # CuPy: Nelder-Mead (CPU) con la loss (misma que la evolución) evaluada en GPU
        if not has("scipy"):
            if verbose: print("SciPy no disponible: omitiendo fine-tuning (backend CuPy).")
            return expr
        x_d = to_device(np.asarray(X).reshape(-1,1)); y_d = to_device(np.asarray(Y).reshape(-1,1))
        dy_d, ddy_d = _derivs_to_device() if USE_DERIV_LOSS else (None, None)
        x0 = np.array([n.value for n in nodes], dtype=np.float64)
        from scipy.optimize import minimize
        def loss_fn(cvec):
            c_map = {nodes[i]._id: to_device(np.array([cvec[i]], dtype=np.float32)) for i in range(len(nodes))}
            yhat = expr.eval_backend(x_d, c_map)
            if cp.isnan(yhat).any() or cp.isinf(yhat).any(): return 1e12
            loss = LAMBDA_Y*cp.mean((yhat - y_d)**2)
            if (dy_d is not None) or (ddy_d is not None):
                # Derivadas por diferencias finitas (coherente con mse_expr_with_derivs)
                h = cp.asarray(1e-3, dtype=x_d.dtype)
                fph = expr.eval_backend(x_d + h, c_map)
                fmh = expr.eval_backend(x_d - h, c_map)
                if dy_d is not None:
                    loss = loss + LAMBDA_Yp*cp.mean(((fph - fmh)/(2*h) - dy_d)**2)
                if ddy_d is not None:
                    loss = loss + LAMBDA_Ypp*cp.mean(((fph - 2*yhat + fmh)/(h*h) - ddy_d)**2)
            return float(loss.get())
        res = minimize(loss_fn, x0, method="Nelder-Mead", options={"maxiter": max(2000, 20*len(x0))})
        if verbose: print("Nelder-Mead:", res.message, "  loss:", res.fun)
        for i,n in enumerate(nodes): n.value=float(res.x[i])
        return expr

def _const_candidates(v):
    """Candidatos 'bonitos' para una constante: entero cercano, fracción simple,
    múltiplos/combinaciones de pi y E (via sympy.nsimplify)."""
    out = []
    r = round(v)
    if abs(v - r) < 0.05: out.append(float(r))
    try:
        fr = sp.Rational(v).limit_denominator(12)
        out.append(float(fr))
    except Exception:
        pass
    try:
        cand = sp.nsimplify(v, [sp.pi, sp.E], tolerance=0.01, rational=False)
        if cand.is_number: out.append(float(cand))
    except Exception:
        pass
    uniq = []
    for c in out:
        if np.isfinite(c) and abs(c - v) < 0.05*max(1.0, abs(v)) and all(c != u for u in uniq):
            uniq.append(c)
    return uniq

def recognize_constants(expr, X, Y, alpha=1e-3, verbose=False):
    """Fase 3.2: reconocimiento de constantes tras el fine-tuning
    (2.9999*sin(x) -> 3*sin(x)); cada sustitución se acepta solo si el
    fitness sobre los datos completos NO empeora."""
    x_d = to_device(np.asarray(X).reshape(-1,1)); y_d = to_device(np.asarray(Y).reshape(-1,1))
    e2 = expr.clone()
    base = fitness(e2, x_d, y_d, alpha=alpha)
    for c in collect_constants(e2):
        for cand in _const_candidates(c.value):
            if cand == c.value: continue
            old = c.value; c.value = cand
            f = fitness(e2, x_d, y_d, alpha=alpha)
            if f <= base + 1e-12:
                base = f
                if verbose: print(f"  constante {old:.8g} -> {cand:.8g}  (fit {f:.6g})")
                break
            c.value = old
    return e2

def fine_tune_hof(hof, X, Y, steps=250, lr=5e-3, verbose=True, alpha=1e-3):
    """Fase 2.4: ajusta las constantes de TODAS las expresiones del Hall of Fame
    en paralelo con un único optimizador Adam (pérdida sumada, parámetros
    disjuntos => equivalente a optimizarlas por separado pero con un solo
    backward por paso). Devuelve un HallOfFame reordenado por fitness."""
    if BACKEND != "torch":
        new_hof = HallOfFame(k=hof.k)
        x_d = to_device(np.asarray(X).reshape(-1,1)); y_d = to_device(np.asarray(Y).reshape(-1,1))
        for f, e in hof.items:
            e2 = fine_tune_constants(e.clone(), X, Y, steps=steps, lr=lr, verbose=False)
            new_hof.update(e2, fitness(e2, x_d, y_d, alpha=alpha))
        return new_hof

    x_t = to_device(np.asarray(X).reshape(-1,1)); y_t = to_device(np.asarray(Y).reshape(-1,1))
    dy_t, ddy_t = _derivs_to_device() if USE_DERIV_LOSS else (None, None)
    entries = []; all_params = []
    for f, e in hof.items:
        e = e.clone()
        nodes = collect_constants(e)
        params = {c._id: torch.tensor([c.value], dtype=x_t.dtype, device=x_t.device, requires_grad=True)
                  for c in nodes}
        entries.append((e, nodes, params))
        all_params += list(params.values())
    if not all_params:
        return hof

    opt = torch.optim.Adam(all_params, lr=lr)
    for i in range(1, steps+1):
        opt.zero_grad(set_to_none=True)
        total = None
        for e, nodes, params in entries:
            l = _ft_loss_torch(e, params, x_t, y_t, dy_t, ddy_t)
            total = l if total is None else total + l
        total.backward(); opt.step()
        if verbose and (i%50==0 or i==1): print(f"FT-HOF paso {i:4d}  loss_total={total.item():.6g}")

    new_hof = HallOfFame(k=hof.k)
    for e, nodes, params in entries:
        for c in nodes:
            v = float(params[c._id].detach().cpu().item())
            if np.isfinite(v): c.value = v   # si Adam divergió, conserva el valor previo
        new_hof.update(e, fitness(e, x_t, y_t, alpha=alpha))
    return new_hof


## Modo EML puro (Fase 4.2)
Gramática `S -> hoja_afin | eml(S, S)` con hojas `alpha_i + beta_i*x` entrenadas
por gradiente en GPU. Los multi-starts van POR LOTES (una dimensión extra del
tensor, no un bucle) porque la init aleatoria falla a profundidad > 4 (paper,
sec. 4.3); la evolución genética con catálogo eml-only sirve de inicializador.


In [ ]:

def _eml_full_structure(depth):
    """Árbol eml completo de profundidad `depth`; devuelve (spec, n_hojas).
    spec anidado: ("eml", izq, der) | ("leaf", i)."""
    counter = [0]
    def rec(d):
        if d <= 0:
            i = counter[0]; counter[0] += 1
            return ("leaf", i)
        return ("eml", rec(d-1), rec(d-1))
    return rec(depth), counter[0]

def _eml_structure_from_tree(tree, x_d, max_depth=None):
    """Convierte un árbol del motor en estructura eml pura: los nodos Eml se
    conservan (hasta max_depth) y cualquier otro subárbol se aproxima por una
    hoja afín a + b*x ajustada por mínimos cuadrados sobre x_d (init del modo puro)."""
    leaf_inits = []
    xcol = x_d.reshape(-1)
    n = float(xcol.shape[0])
    def rec(nd, d=0):
        if isinstance(nd, Eml) and (max_depth is None or d < max_depth):
            return ("eml", rec(nd.o1, d+1), rec(nd.o2, d+1))
        with torch.no_grad():
            v = nd.eval_backend(x_d).reshape(-1)
            v = torch.nan_to_num(v, nan=0.0, posinf=1e6, neginf=-1e6)
            sx = xcol.sum(); sxx = (xcol*xcol).sum()
            sv = v.sum(); sxv = (xcol*v).sum()
            den = n*sxx - sx*sx
            b = (n*sxv - sx*sv)/den
            a = (sv - b*sx)/n
        leaf_inits.append((float(a.item()), float(b.item())))
        return ("leaf", len(leaf_inits)-1)
    return rec(tree), leaf_inits

def eml_pure_train(spec, n_leaves, X, Y, n_starts=256, steps=1500, lr=5e-3,
                   leaf_init=None, init_fraction=0.25, jitter=0.3,
                   verbose=True, use_compile=True):
    """Entrena las hojas afines de una estructura eml FIJA con n_starts
    reinicios en paralelo (tensores (N, n_hojas)); un solo Adam y una pérdida
    (N,) sumada. Devuelve (árbol del motor con las hojas ganadoras, mse)."""
    assert BACKEND == "torch", "el modo EML puro requiere backend torch"
    x_t = to_device(np.asarray(X).reshape(-1)); y_t = to_device(np.asarray(Y).reshape(-1))
    N = int(n_starts); dev = x_t.device
    # init pequeña: las cadenas de exp anidadas divergen con hojas grandes
    A = torch.empty((N, n_leaves), dtype=x_t.dtype, device=dev).uniform_(-1.5, 1.5)
    B = torch.empty((N, n_leaves), dtype=x_t.dtype, device=dev).uniform_(-1.5, 1.5)
    if leaf_init:
        a0 = torch.tensor([a for a, b in leaf_init], dtype=x_t.dtype, device=dev)
        b0 = torch.tensor([b for a, b in leaf_init], dtype=x_t.dtype, device=dev)
        k = max(1, int(N*init_fraction))
        A[:k] = a0[None, :] + jitter*torch.randn(k, n_leaves, dtype=x_t.dtype, device=dev)
        B[:k] = b0[None, :] + jitter*torch.randn(k, n_leaves, dtype=x_t.dtype, device=dev)
        A[0] = a0; B[0] = b0   # un reinicio exacto con la init evolutiva
    A.requires_grad_(True); B.requires_grad_(True)

    def forward():
        def rec(node):
            if node[0] == "leaf":
                i = node[1]
                return A[:, i:i+1] + B[:, i:i+1]*x_t[None, :]
            l = rec(node[1]); r = rec(node[2])
            # eml con guardas suaves y diferenciables; el recorte a 20 (exp<=5e8)
            # mantiene la pérdida finita para que el gradiente siga fluyendo
            return torch.exp(torch.clamp(l, max=20.0)) - torch.log(torch.abs(r) + 1e-8)
        yh = rec(spec)                                       # (N, n_puntos)
        return torch.mean((yh - y_t[None, :])**2, dim=1)     # (N,)

    fwd = forward
    if use_compile:
        try:
            fwd = torch.compile(forward)   # estructura fija => compila UNA vez (Fase 2.3)
        except Exception:
            fwd = forward
    opt = torch.optim.Adam([A, B], lr=lr)
    reseed_every = max(1, steps//4)
    for i in range(1, steps+1):
        opt.zero_grad(set_to_none=True)
        losses = fwd()
        loss = torch.nan_to_num(losses, nan=1e12, posinf=1e12).sum()  # params disjuntos: suma = por-reinicio
        loss.backward()
        opt.step()
        # re-siembra por lotes: los reinicios divergidos se relanzan desde el mejor + ruido
        if i % reseed_every == 0 and i < steps:
            with torch.no_grad():
                lf = torch.nan_to_num(losses, nan=1e12, posinf=1e12)
                bad = lf > max(1e6, 100.0*float(lf.min().item()) + 1e6)
                if bad.any() and (~bad).any():
                    jb = int(torch.argmin(lf).item())
                    nb = int(bad.sum().item())
                    A[bad] = A[jb][None, :] + 0.3*torch.randn(nb, n_leaves, dtype=x_t.dtype, device=dev)
                    B[bad] = B[jb][None, :] + 0.3*torch.randn(nb, n_leaves, dtype=x_t.dtype, device=dev)
        if verbose and (i % 300 == 0 or i == 1):
            print(f"EML-pure paso {i:4d}  mejor MSE={float(torch.nan_to_num(losses, nan=1e12).min().item()):.6g}")

    with torch.no_grad():
        losses = torch.nan_to_num(forward(), nan=1e12, posinf=1e12)
        j = int(torch.argmin(losses).item())
        best_mse = float(losses[j].item())
    Aj = A[j].detach().cpu().numpy(); Bj = B[j].detach().cpu().numpy()

    def build(node):
        if node[0] == "leaf":
            i = node[1]
            return Add(Constant(float(Aj[i])), Mul(Constant(float(Bj[i])), FloatInput()))
        return Eml(build(node[1]), build(node[2]))
    return build(spec), best_mse

def eml_pure_fit(X, Y, depth=None, n_starts=None, steps=None, lr=None,
                 use_evolution_init=True, evo_generations=60, evo_pop=100, verbose=True):
    """Modo EML puro completo: estructura desde una evolución corta con catálogo
    eml-only (inicializador de pesos, Fase 4.2) o árbol completo de profundidad
    EML_PURE_DEPTH; después, multi-starts por lotes con Adam."""
    depth    = depth    if depth    is not None else EML_PURE_DEPTH
    n_starts = n_starts if n_starts is not None else EML_PURE_STARTS
    steps    = steps    if steps    is not None else EML_PURE_STEPS
    lr       = lr       if lr       is not None else EML_PURE_LR

    spec = None; leaf_init = None
    if use_evolution_init:
        cat_un_e, cat_bin_e = build_catalogs([], ["eml"])
        best_e, _f, _h, _a, _hof = evolve(np.asarray(X), np.asarray(Y), cat_un_e, cat_bin_e,
                                          pop_size=evo_pop, generations=evo_generations,
                                          max_depth=depth, verbose=False,
                                          use_macros=False, use_hint=False, hof_k=5)
        x_d = to_device(np.asarray(X).reshape(-1,1))
        # tope holgado: evita estructuras patológicas sin recortar la expresividad
        spec_e, leaf_init_e = _eml_structure_from_tree(best_e, x_d, max_depth=max(2*depth, 8))
        if spec_e[0] == "eml":   # si el mejor evolutivo no contenía eml, se ignora
            spec, leaf_init = spec_e, leaf_init_e
            if verbose: print(f"init evolutiva: estructura con {len(leaf_init)} hojas  ({best_e.to_text()[:70]}...)")
    if spec is None:
        spec, n_leaves = _eml_full_structure(depth)
        leaf_init = None
    else:
        n_leaves = len(leaf_init)
    return eml_pure_train(spec, n_leaves, X, Y, n_starts=n_starts, steps=steps, lr=lr,
                          leaf_init=leaf_init, verbose=verbose)


## Exportación a SymPy (conversión recursiva) + test de ida y vuelta


In [ ]:

# --- Conversión a SymPy (Fase 1.1): recursiva nodo -> SymPy, sin str.replace ---
def to_sympy(best):
    x = sp.Symbol('x', real=True)
    return best.to_sympy(x), x

def check_sympy_roundtrip(expr, lo=-3.0, hi=3.0, n=64, rtol=1e-3, atol=1e-3, min_cover=0.5):
    """Test de ida y vuelta: la expresión SymPy y la del motor deben coincidir
    numéricamente en una malla de puntos. Devuelve True/False, o None si las
    guardas numéricas del motor (p.ej. pow con base negativa) dejan demasiados
    puntos sin comparar (resultado no concluyente)."""
    f_sym, xs = to_sympy(expr)
    grid = np.linspace(lo, hi, n).astype(np.float64)
    f_num = sp.lambdify(xs, f_sym, modules=["scipy", "numpy"])
    with np.errstate(all="ignore"):
        try:
            y_sym = np.asarray(f_num(grid), dtype=np.complex128) * np.ones_like(grid, dtype=np.complex128)
        except Exception:
            return None
    y_eng = to_numpy(expr.eval_backend(to_device(grid.reshape(-1,1)))).reshape(-1).astype(np.float64)
    es_real = np.abs(y_sym.imag) < 1e-9
    y_real = y_sym.real
    mask = es_real & np.isfinite(y_real) & np.isfinite(y_eng) & (np.abs(y_eng) < 1e6)
    if mask.sum() < n*min_cover:
        return None  # cobertura insuficiente para validar
    return bool(np.allclose(y_real[mask], y_eng[mask], rtol=rtol, atol=atol))


## Demo rápida (puedes reemplazar con tus datos)
- Cambia `BACKEND` arriba si quieres CuPy.
- Activa `USE_DERIV_LOSS` y llena `DY`, `DDY` si las tienes.
- Ajusta `curriculum`, `sample_schedule`, `unary_priors`, `binary_priors`.


In [ ]:

# === Datos de prueba ===
X = np.linspace(-5, 5, 400).astype(np.float32)
Y_true = 2*X**3 + 3*np.sin(X) + 1
rng = np.random.default_rng(0)
#Y = (Y_true + rng.normal(0, 0.25, size=Y_true.shape)).astype(np.float32)
Y = (Y_true ).astype(np.float32)

# # Derivadas (opcionales) para acelerar
if USE_DERIV_LOSS:
    DY = (6*X**2 + 3*cos(X)).astype(np.float32)    # y' real de la función de prueba
    DDY= (12*X - 3*sin(X)).astype(np.float32)       # y'' real

# === Hiperparámetros ===
# Con ISLANDS=4 y GEN=250 (generaciones por isla) el caso de prueba se recupera
# 5/5 semillas en ~2 min de reloj (medido en RTX 4090); con una sola población
# usa GEN=1000. La Fase 2 dejó la GPU libre para permitirse estos defaults.
POP=160; GEN=250; DEPTH=5; ELITE=10; PMUT=0.6; PXOVER=0.3; ALPHA=1e-3
FINE_TUNE=True; FT_STEPS=250; FT_LR=5e-3
FT_HOF=True   # Fase 2.4: ajusta todo el Hall of Fame en paralelo (False = solo el mejor, como antes)

# Fase 3: calidad de búsqueda
VAL_FRACTION=0.2         # split train/validación; el mejor final se elige por validación (0 = off)
RESTART_PATIENCE=80      # reinicio con diversidad fresca tras N gens sin mejora (0 = off)
EARLY_STOP_PATIENCE=300  # corta la corrida tras N gens sin mejora (0 = off)
RECOGNIZE_CONSTS=True    # redondeo + nsimplify de constantes tras el fine-tuning
ISLANDS=4                # >1 = islas paralelas con migración (evolve_islands); 0/1 = una población

# AOS
AOS_CONF = dict(tau=0.8, lr=0.4, decay=0.98, prune_every=40, min_keep_un=3, min_keep_bin=3, prune_threshold=0.04)

# Priors: incentiva operadores coherentes con tu conocimiento previo
unary_priors = {"sin":0.6, "cos":0.1, "log":0.1}
binary_priors = {"add":0.3, "mul":0.1, "pow":0.4}

# Submuestreo por generaciones: (gen_limite, fraccion)
sample_schedule = [(40, 0.25), (90, 0.6), (130, 0.85), (GEN, 1.0)]

# Currículum de complejidad por fases (con EML_MODE="operator", eml entra en la fase final)
_bin_final = ["add","sub","mul","div","pow"] + (["eml"] if EML_MODE == "operator" else [])
curriculum = [
    {"until": 40, "depth": 3, "un": ["sin","cos","abs","neg","sqrt"], "bin": ["add","sub","mul"]},
    {"until": 90, "depth": 4, "un": ["sin","cos","log","sqrt","abs","neg"], "bin": ["add","sub","mul","div"]},
    {"until": GEN, "depth": 5, "un": ["sin","cos","log","sqrt","abs","neg"], "bin": _bin_final},
]

# === Evolución ===
t0=time.time()
_common = dict(pop_size=POP, generations=GEN, max_depth=DEPTH,
               elite=ELITE, p_mut=PMUT, p_xover=PXOVER, alpha=ALPHA,
               aos_params=AOS_CONF,
               use_macros=True, p_macro=0.25,
               use_hint=True, hint_fraction=0.35, hint_jitter=0.25,
               unary_priors=unary_priors, binary_priors=binary_priors,
               sample_schedule=sample_schedule, curriculum=curriculum,
               hill_climb_steps=3, hof_k=10,
               val_fraction=VAL_FRACTION,
               early_stop_patience=EARLY_STOP_PATIENCE,
               restart_patience=RESTART_PATIENCE)
if ISLANDS and ISLANDS > 1:
    _common.pop("pop_size"); _common.pop("generations"); _common.pop("val_fraction")
    best, fit, hist, aos, hof = evolve_islands(
        X, Y, CAT_UN, CAT_BIN, n_islands=ISLANDS, migrate_every=50, migrants=3,
        generations=GEN, pop_size=POP, verbose=True, val_fraction=VAL_FRACTION, **_common)
else:
    best, fit, hist, aos, hof = evolve(X, Y, CAT_UN, CAT_BIN, verbose=True, **_common)
t1=time.time()

if FINE_TUNE:
    if FT_HOF:
        # Ajusta las constantes de todo el Hall of Fame en paralelo (Fase 2.4)
        hof = fine_tune_hof(hof, X, Y, steps=FT_STEPS, lr=FT_LR, verbose=True, alpha=ALPHA)
        fit, best = hof.items[0]
        best = best.clone()
    else:
        best = fine_tune_constants(best, X, Y, steps=FT_STEPS, lr=FT_LR, verbose=True)

if RECOGNIZE_CONSTS:
    # Fase 3.2: reconoce enteros/fracciones/pi/E si el fitness no empeora
    best = recognize_constants(best, X, Y, alpha=ALPHA, verbose=True)
t2=time.time()

print("\n=== HALL OF FAME (k mejores de toda la corrida, datos completos) ===")
for f_h, e_h in hof.items:
    _vm = hof.val_mse.get(e_h.to_text()) if getattr(hof, "val_mse", None) else None
    extra = f"  val_mse={_vm:.4g}" if _vm is not None else ""
    print(f"  fit={f_h:.6g}{extra}   {e_h.to_text()}")

if getattr(hof, "pareto", None):
    print("\n=== FRENTE DE PARETO precisión-complejidad (SELECTION='nsga2') ===")
    for _m, _n, _e in hof.pareto:
        print(f"  mse={_m:.6g}  nodos={_n:3d}   {_e.to_text()}")

print("\n=== RESULTADO ===")
print("Mejor expresión:", best.to_text())
if BACKEND=="torch":
    x_d = to_device(X.reshape(-1,1)); y_d = to_device(Y.reshape(-1,1))
    mse_final = float(torch.mean((best.eval_backend(x_d)-y_d)**2).item())
else:
    x_d = to_device(X.reshape(-1,1)); y_d = to_device(Y.reshape(-1,1))
    mse_final = float(cp.mean((best.eval_backend(x_d)-y_d)**2).get())
print("MSE final:", mse_final)
print(f"Tiempo evolución: {t1-t0:.2f}s  |  total: {t2-t0:.2f}s")

# Curva
xplot = np.linspace(float(X.min()), float(X.max()), 700).astype(np.float32)
x_d = to_device(xplot.reshape(-1,1))
yhat = to_numpy(best.eval_backend(x_d)).reshape(-1)

plt.figure(figsize=(8.5,4.8))
plt.scatter(X, Y, s=14, alpha=0.7, label="Datos")
plt.plot(xplot, yhat, linewidth=2, label="Expresión encontrada")
plt.xlabel("x"); plt.ylabel("y"); plt.title(f"Symbolic Regression — AOS+Accelerators ({BACKEND})")
plt.legend(); plt.grid(True); plt.show()

# Probabilidades finales aproximadas (softmax de scores)
def show_probs(title, scores_dict, tau):
    names = list(scores_dict.keys())
    if not names:
        print(title, " (catálogo vacío)"); return
    v = np.array([scores_dict[n] for n in names], dtype=np.float64)
    v = v - v.max()
    p = np.exp(v / max(1e-6, tau)); p = p / p.sum()
    order = np.argsort(-p)
    print(title)
    for i in order:
        print(f"  {names[i]:>16s}: {p[i]:.3f}")
show_probs("Probabilidades unarias (finales):", aos.un_scores, AOS_CONF['tau'])
show_probs("Probabilidades binarias (finales):", aos.bin_scores, AOS_CONF['tau'])


## Modo EML puro (se activa con EML_MODE="pure")


In [ ]:
if EML_MODE == "pure":
    from IPython.display import display
    t_eml0 = time.time()
    best_eml, mse_eml = eml_pure_fit(X, Y, verbose=True)
    t_eml1 = time.time()
    print("\n=== EML PURO ===")
    print("Expresión:", best_eml.to_text()[:300])
    print(f"MSE={mse_eml:.6g}   tiempo={t_eml1-t_eml0:.1f}s")
    # Ambas formas simbólicas (Fase 4.3): compacta eml(u,v) y expandida exp/ln
    EML_SYMPY_EXPAND = False
    f_eml_c, _xs = to_sympy(best_eml)
    display(sp.Eq(sp.Symbol('f_eml'), f_eml_c))
    EML_SYMPY_EXPAND = True
    f_eml_e, _xs = to_sympy(best_eml)
    display(sp.Eq(sp.Symbol('f_eml_exp'), f_eml_e))
    if mse_eml < globals().get("mse_final", float("inf")):
        print("El modo EML puro superó al mejor evolutivo en este caso.")
else:
    print("EML_MODE =", EML_MODE, "(pon EML_MODE='pure' en la primera celda para activar esta sección)")

In [ ]:
from IPython.display import display

# --- Uso de la conversión recursiva a SymPy ---
f_sym, x = to_sympy(best)

# Test de ida y vuelta: motor y SymPy deben coincidir numéricamente (Fase 1.1)
_rt = check_sympy_roundtrip(best, lo=float(X.min()), hi=float(X.max()))
if _rt is None:
    print("Roundtrip motor <-> SymPy: no concluyente (guardas numéricas activas en parte del dominio)")
else:
    assert _rt, "Roundtrip motor <-> SymPy falló"
    print("Roundtrip motor <-> SymPy: OK")

display(sp.Eq(sp.Symbol('f(x)'), f_sym))          # Muestra f(x)
display(sp.Eq(sp.Symbol("f'(x)"), sp.diff(f_sym, x)))   # Primera derivada
display(sp.Eq(sp.Symbol("∫f(x)dx"), sp.integrate(f_sym, x)))  # Integral simbólica

In [ ]:
# === GPU-ready: SymPy -> (CuPy|NumPy) + gráficas ===
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Intentar CuPy
try:
    import cupy as cp
    from cupyx.scipy import special as cpsp
    HAS_CUPY = True
except Exception:
    HAS_CUPY = False

# ----- helpers robustos -----
def safe_derivative(expr, var, order=1):
    d = sp.diff(expr, var, order)
    d = sp.simplify(d.doit())
    # La derivada de |u| introduce DiracDelta (distribucional): se anula en
    # casi todo punto, así que la eliminamos para poder evaluar/graficar
    d = d.replace(sp.DiracDelta, lambda *a: sp.S.Zero)
    return d

# Mapeos NumPy (CPU)
NUMPY_LOCALS = {
    'sin': np.sin, 'cos': np.cos, 'tan': np.tan,
    'log': np.log, 'sqrt': np.sqrt, 'abs': np.abs, 'Abs': np.abs, 'exp': np.exp,
    'asin': np.arcsin, 'acos': np.arccos, 'atan': np.arctan,
    'Min': np.minimum, 'Max': np.maximum, 'sign': np.sign
}

# Mapeos CuPy (GPU)
if HAS_CUPY:
    CUPY_LOCALS = {
        'sin': cp.sin, 'cos': cp.cos, 'tan': cp.tan,
        'log': cp.log, 'sqrt': cp.sqrt, 'abs': cp.abs, 'Abs': cp.abs, 'exp': cp.exp,
        'asin': cp.arcsin, 'acos': cp.arccos, 'atan': cp.arctan,
        'Min': cp.minimum, 'Max': cp.maximum, 'sign': cp.sign,
        # especiales (si aparecen en tu expresión)
        'erf': cpsp.erf, 'erfc': cpsp.erfc, 'gamma': cpsp.gamma,
        'j0': cpsp.j0, 'j1': cpsp.j1, 'y0': cpsp.y0, 'y1': cpsp.y1
    }

def safe_lambdify_gpu(var, expr):
    """
    Devuelve una función vectorizada que opera en GPU (CuPy) si está disponible.
    Si falla (o no hay CuPy), usa NumPy (CPU) como fallback.
    """
    if HAS_CUPY:
        try:
            # Solo CuPy: todas las ufuncs en GPU
            return sp.lambdify(var, expr, modules=[CUPY_LOCALS])
        except Exception:
            # Mezcla: intenta con CuPy y, si algo no está mapeado, deja que SymPy evalúe
            try:
                return sp.lambdify(var, expr, modules=[CUPY_LOCALS, "sympy"])
            except Exception:
                pass
    # Fallback a NumPy (rápido en CPU)
    try:
        return sp.lambdify(var, expr, modules=[NUMPY_LOCALS, "numpy"])
    except Exception:
        # Último recurso: NumPy + SymPy
        return sp.lambdify(var, expr, modules=["numpy", "sympy"])

# ----- convertir 'best' a SymPy -----
f_sym, x = to_sympy(best)

# ----- derivadas simbólicas seguras -----
f1_sym = safe_derivative(f_sym, x, 1)
f2_sym = safe_derivative(f_sym, x, 2)

# ----- mostrar en notación bonita -----
display(sp.Eq(sp.Symbol('f(x)'), f_sym))
display(sp.Eq(sp.Symbol("f'(x)"), f1_sym))
display(sp.Eq(sp.Symbol("f''(x)"), f2_sym))

# ----- funciones numéricas: GPU si se puede, CPU (printer NumPy) de respaldo -----
# El printer NumPy real maneja Piecewise/sign/Min/Max; los dicts de módulos no.
f_gpu  = safe_lambdify_gpu(x, f_sym)  if HAS_CUPY else None
f1_gpu = safe_lambdify_gpu(x, f1_sym) if HAS_CUPY else None
f2_gpu = safe_lambdify_gpu(x, f2_sym) if HAS_CUPY else None
f_cpu  = sp.lambdify(x, f_sym,  modules=["scipy", "numpy"])
f1_cpu = sp.lambdify(x, f1_sym, modules=["scipy", "numpy"])
f2_cpu = sp.lambdify(x, f2_sym, modules=["scipy", "numpy"])

# Si la expresión (o su derivada) es constante, lambdify devuelve un escalar:
# lo difundimos a la forma de X para poder graficar
def _a1d(v):
    a = np.asarray(v, dtype=np.float64).reshape(-1)
    return np.full(X.shape, a[0]) if a.size == 1 else a

def _eval_pred(fg, fc):
    """Evalúa en GPU si es posible; ante cualquier fallo, CPU con printer NumPy."""
    if fg is not None:
        try:
            return _a1d(cp.asnumpy(cp.asarray(fg(cp.asarray(X)))))
        except Exception:
            pass
    try:
        with np.errstate(all="ignore"):
            return _a1d(fc(X.astype(np.float64)))
    except Exception as e:
        print("Aviso: no se pudo evaluar numéricamente esta expresión:", type(e).__name__)
        return np.full(X.shape, np.nan)

Y_pred   = _eval_pred(f_gpu,  f_cpu)
DY_pred  = _eval_pred(f1_gpu, f1_cpu)
DDY_pred = _eval_pred(f2_gpu, f2_cpu)

Y_true_plot  = 2*X**3 + 3*np.sin(X) + 1
DY_true_plot = 6*X**2 + 3*np.cos(X)
DDY_true_plot= 12*X - 3*np.sin(X)

# ----- graficar comparativa f, f', f'' -----
plt.figure(figsize=(12, 8))

# f(x)
plt.subplot(3,1,1)
plt.plot(X, Y_true_plot, label='f real')
plt.plot(X, Y_pred, label='f encontrada')
plt.title("Comparación de f(x)")
plt.grid(alpha=0.3); plt.legend()

# f'(x)
plt.subplot(3,1,2)
plt.plot(X, DY_true_plot, label="f'(x) real")
plt.plot(X, DY_pred, label="f'(x) encontrada")
plt.title("Comparación de f'(x)")
plt.grid(alpha=0.3); plt.legend()

# f''(x)
plt.subplot(3,1,3)
plt.plot(X, DDY_true_plot, label="f''(x) real")
plt.plot(X, DDY_pred, label="f''(x) encontrada")
plt.title("Comparación de f''(x)")
plt.grid(alpha=0.3); plt.legend()

plt.tight_layout()
plt.show()